# Capstone 4 — Coding Agents for a Team (Claude Code and Codex) · Practice

**Goal:** configure one sample repository so that Claude Code **and** Codex share the same conventions, skills, MCP servers and guardrails, run both as CI reviewers with structured output, and ship one change that is verified with evidence a reviewer can check.

- **Brief and theory:** [README.md](README.md). This notebook holds the code; the design reasoning lives in the README.
- **Estimated time:** 5–7 hours
- **Estimated API cost:** **$0** with the defaults (`RUN_CLAUDE = False`, `RUN_CODEX = False`): every agent output is a realistic **sample**. With live mode on, the handful of capped CLI calls cost roughly $0.50–$2 in total with the default models. That figure is a rough guess, not a measurement; each Claude call is capped with `--max-budget-usd`, so the cap, not the guess, is your guarantee.
- **Prerequisites:** the repo venv from [00 — Prerequisites](../../../00-prerequisites/README.md); Claude Chapter 5 and OpenAI Chapter 5. For live cells: Claude Code and/or the Codex CLI installed and signed in, plus `git`.

**How this notebook works**

- Everything is built in a **temporary directory**: a sample repo called `kestrel-shop` and a **fake home directory** for personal config (`~/.claude.json`, `~/.codex/config.toml`). Your real `~/.claude` and `~/.codex` are never read or written.
- Offline cells parse and check every config file the way the tools would (YAML frontmatter, JSON, TOML, the Starlark `.rules` file, glob patterns, permission precedence, MCP scope resolution).
- The sample repo's own `scripts/verify.py` **really runs** (standard library only), so the verification evidence in section 4.10 is real.
- Live cells call `claude -p` or `codex exec` only when you set `RUN_CLAUDE = True` / `RUN_CODEX = True` **and** the CLI is on your `PATH`. Otherwise they print the exact command and use a sample output.

> **Safety note:** `claude -p` loads a folder's hooks, `.mcp.json` servers and `CLAUDE.md` without a trust prompt, and `codex exec` can edit files when you give it `workspace-write`. Only run live cells against the sandbox repo this notebook creates.

In [ ]:
import ast
import difflib
import hashlib
import inspect
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import tempfile
import textwrap
import tomllib
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv

try:
    import yaml  # PyYAML is installed with JupyterLab (requirements.txt); not a direct dependency
except ImportError:  # pragma: no cover
    yaml = None
    print("PyYAML is missing: `pip install pyyaml` (it normally ships with JupyterLab). YAML checks will fail.")

load_dotenv(find_dotenv())

CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

RUN_CLAUDE = False   # True = real `claude -p` calls (costs money, needs Claude Code)
RUN_CODEX = False    # True = real `codex exec` calls (costs money, needs the Codex CLI)
CLAUDE_BIN = shutil.which("claude")
CODEX_BIN = shutil.which("codex")
MAX_BUDGET_USD = "0.50"  # cap for every live Claude call

print(f"Claude model={CLAUDE_MODEL}  CLI: {CLAUDE_BIN or 'not on PATH'}  RUN_CLAUDE={RUN_CLAUDE}")
print(f"OpenAI model={OPENAI_MODEL}  CLI: {CODEX_BIN or 'not on PATH'}  RUN_CODEX={RUN_CODEX}")

In [ ]:
# Live prerequisites. They are only enforced when you opt in, so offline cells keep working.
if RUN_CLAUDE:
    assert CLAUDE_BIN, "Install Claude Code and put `claude` on PATH (see 00-prerequisites)"
if RUN_CODEX:
    assert CODEX_BIN, "Install the Codex CLI and put `codex` on PATH (see openai-codex/05-codex)"
if RUN_CLAUDE or RUN_CODEX:
    assert shutil.which("git"), "Live runs need git: both CLIs expect a Git repository"
    print("Live mode: real CLI calls will run against the sandbox repo only.")
else:
    print("Offline mode: live cells print the command they would run and use sample output.")

## 0. Build the sample repository

`kestrel-shop` is deliberately small: a pricing function, one API handler, tests in **two** places (`tests/` and next to the code), one SQL migration, and a `scripts/verify.py` that compiles the package and runs every `test_*.py` with the standard library. It also contains one real bug that section 4.10 fixes: an unknown discount code raises `KeyError`.

In [ ]:
ROOT = Path(tempfile.mkdtemp(prefix="kestrel-shop-"))
HOME = Path(tempfile.mkdtemp(prefix="fake-home-"))   # stands in for ~ (personal config)


def write(rel: str, text: str, base: Path = ROOT) -> Path:
    """Write a file under base, creating parent folders; returns the path."""
    path = base / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8")
    return path


def link(rel: str, target_rel: str, base: Path = ROOT) -> str:
    """Create a relative symlink rel -> target_rel. Falls back to a copy where symlinks are unavailable."""
    path = base / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    rel_target = os.path.relpath(base / target_rel, path.parent)
    try:
        if path.is_symlink() or path.exists():
            shutil.rmtree(path) if path.is_dir() and not path.is_symlink() else path.unlink()
        path.symlink_to(rel_target, target_is_directory=(base / target_rel).is_dir())
        return "symlink"
    except OSError:  # e.g. Windows without Developer Mode
        src = base / target_rel
        shutil.copytree(src, path) if src.is_dir() else shutil.copyfile(src, path)
        return "copy"


def tree(base: Path = ROOT, skip: tuple[str, ...] = ("__pycache__",)) -> None:
    for p in sorted(base.rglob("*")):
        if any(s in p.parts for s in skip):
            continue
        rel = p.relative_to(base)
        arrow = f" -> {os.readlink(p)}" if p.is_symlink() else ""
        print("    " * (len(rel.parts) - 1) + rel.name + ("/" if p.is_dir() and not p.is_symlink() else "") + arrow)


print("repo:", ROOT)
print("fake home:", HOME)

In [ ]:
write("src/shop/__init__.py", '"""kestrel-shop: pricing and orders."""\n')
write("src/shop/api/__init__.py", "")
write("src/shop/pricing.py", '''
    """Price calculations. Money is always integer cents."""

    DISCOUNT_PERCENT = {"SAVE10": 10, "SAVE25": 25}


    def apply_discount(price_cents: int, code: str | None = None) -> int:
        """Return the price after applying a discount code (case-insensitive)."""
        if price_cents < 0:
            raise ValueError("price_cents must be >= 0")
        if not code:
            return price_cents
        percent = DISCOUNT_PERCENT[code.upper()]
        return price_cents - (price_cents * percent) // 100
''')
write("src/shop/api/routes.py", '''
    """HTTP-agnostic API handlers. Errors are returned, never raised."""
    from shop.pricing import apply_discount


    def quote(payload: dict) -> dict:
        price = payload.get("price_cents")
        if not isinstance(price, int) or price < 0:
            return {"error": {"code": "invalid_request", "message": "price_cents must be a non-negative integer"}}
        return {"total_cents": apply_discount(price, payload.get("code"))}
''')
write("src/shop/api/test_routes.py", '''
    import unittest

    from shop.api.routes import quote


    class QuoteTests(unittest.TestCase):
        def test_quote_ok(self):
            """A valid payload returns the discounted total."""
            self.assertEqual(quote({"price_cents": 1000, "code": "SAVE10"}), {"total_cents": 900})

        def test_quote_rejects_negative_price(self):
            """Bad input returns the error envelope instead of raising."""
            self.assertEqual(quote({"price_cents": -1})["error"]["code"], "invalid_request")
''')
write("tests/__init__.py", "")
write("tests/factories.py", '''
    """Test data builders. Never hardcode order IDs in tests."""
    import itertools

    _ids = itertools.count(1000)


    def make_order(**overrides) -> dict:
        order = {"id": f"ord_{next(_ids)}", "price_cents": 2500, "code": None}
        order.update(overrides)
        return order
''')
write("tests/test_pricing.py", '''
    import unittest

    from shop.pricing import apply_discount
    from tests.factories import make_order


    class PricingTests(unittest.TestCase):
        def test_save10_discount(self):
            """SAVE10 takes 10% off."""
            self.assertEqual(apply_discount(1000, "SAVE10"), 900)

        def test_save25_discount(self):
            """SAVE25 takes 25% off."""
            order = make_order(price_cents=2000, code="SAVE25")
            self.assertEqual(apply_discount(order["price_cents"], order["code"]), 1500)

        def test_zero_price(self):
            """A zero price stays zero with any valid code."""
            self.assertEqual(apply_discount(0, "SAVE10"), 0)

        def test_code_is_case_insensitive(self):
            """Codes match regardless of case."""
            self.assertEqual(apply_discount(1000, "save10"), 900)
''')
write("src/shop/db/migrations/0001_create_orders.sql", '''
    CREATE TABLE orders (id TEXT PRIMARY KEY, price_cents INTEGER NOT NULL, code TEXT);
    -- down: DROP TABLE orders;
''')
write("scripts/verify.py", '''
    """Verify kestrel-shop: compile src/, run every test_*.py, optionally write JSON evidence.

    Usage: python scripts/verify.py [--evidence evidence/verify.json]
    Exit code 0 only when everything compiles, at least one test ran, and nothing failed.
    """
    from __future__ import annotations

    import argparse
    import compileall
    import io
    import json
    import sys
    import time
    import unittest
    from pathlib import Path

    ROOT = Path(__file__).resolve().parents[1]
    SUITES = [("tests", ROOT), ("src", ROOT / "src")]  # (start dir, top-level dir): tests live in both places


    def _ids(suite):
        for item in suite:
            if isinstance(item, unittest.TestSuite):
                yield from _ids(item)
            else:
                yield item.id()


    def main(argv: list[str] | None = None) -> int:
        parser = argparse.ArgumentParser()
        parser.add_argument("--evidence", help="write a JSON evidence file to this path")
        args = parser.parse_args(argv)
        sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
        started = time.time()
        compiled = bool(compileall.compile_dir(str(ROOT / "src"), quiet=1))
        suite = unittest.TestSuite()
        for start, top in SUITES:
            suite.addTests(unittest.TestLoader().discover(str(ROOT / start), pattern="test_*.py", top_level_dir=str(top)))
        test_ids = sorted(_ids(suite))
        out = io.StringIO()
        result = unittest.TextTestRunner(stream=out, verbosity=1).run(suite)
        ok = compiled and result.wasSuccessful() and result.testsRun > 0
        report = {
            "command": "python scripts/verify.py" + (f" --evidence {args.evidence}" if args.evidence else ""),
            "exit_code": 0 if ok else 1,
            "compiled": compiled,
            "tests": {"run": result.testsRun, "failed": len(result.failures),
                      "errors": len(result.errors), "skipped": len(result.skipped)},
            "test_ids": test_ids,
            "failing": sorted(t.id() for t, _ in result.failures + result.errors),
            "duration_s": round(time.time() - started, 3),
        }
        if args.evidence:
            path = ROOT / args.evidence
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text(json.dumps(report, indent=2))
        print(("PASS" if ok else "FAIL") + f": {result.testsRun} tests, {report['tests']['failed']} failed, "
              f"{report['tests']['errors']} errors, compiled={compiled}")
        if not ok:
            print(out.getvalue()[-2000:])
        return report["exit_code"]


    if __name__ == "__main__":
        raise SystemExit(main())
''')
write(".gitignore", '''
    __pycache__/
    .env
    .env.*
    CLAUDE.local.md
    .claude/settings.local.json
    evidence/
''')


def run_verify(root: Path = ROOT, evidence: str | None = "evidence/verify.json", quiet: bool = False) -> dict:
    """Run the sample repo's verify script with this kernel's Python (offline, standard library only)."""
    argv = [sys.executable, "scripts/verify.py"] + (["--evidence", evidence] if evidence else [])
    proc = subprocess.run(argv, cwd=root, capture_output=True, text=True, timeout=120)
    if not quiet:
        print(proc.stdout.strip().splitlines()[0] if proc.stdout.strip() else proc.stderr[-500:])
    return json.loads((root / evidence).read_text()) if evidence else {"exit_code": proc.returncode}


BASELINE = run_verify()
print("baseline tests:", BASELINE["tests"], "exit", BASELINE["exit_code"])

> **Observe:** the baseline is green with 6 tests, even though `apply_discount(1000, "WELCOME")` crashes. No test covers unknown codes yet. That gap is the change you will verify in 4.10.

## 4.1 Instruction layering: one source of truth

`AGENTS.md` holds everything both agents need; `CLAUDE.md` imports it with `@AGENTS.md` and adds only Claude-specific notes. Codex reads `AGENTS.md` directly and treats `@path` as plain text, so `AGENTS.md` itself must not contain imports. [Theory](README.md#41-instruction-layering-one-source-of-truth)

In [ ]:
AGENTS_MD = """
# AGENTS.md: kestrel-shop

Shared instructions for every coding agent. Codex reads this file directly; Claude Code imports it from CLAUDE.md.

## Project
- Python 3.11+, standard library only. Source in `src/shop/`; tests in `tests/` and next to the code (`test_*.py`).
- Money is always integer cents (`int`). Never use `float` for prices.
- API handlers return errors as `{"error": {"code": ..., "message": ...}}` and never raise to the caller.

## Commands
- Verify everything: `python scripts/verify.py` (compiles `src/`, runs every `test_*.py`).
- Verify and write evidence: `python scripts/verify.py --evidence evidence/verify.json`.

## Before you edit (routing)
| When you touch | Read first |
|---|---|
| any `test_*.py` file | `docs/agents/testing.md` |
| `src/shop/api/` | `docs/agents/api.md` |
| `src/shop/db/migrations/` | `docs/agents/migrations.md` |

## Definition of done (verification contract)
- Run `python scripts/verify.py --evidence evidence/verify.json`. A change is done only when it exits 0.
- Never delete, skip or weaken an existing test to make verification pass.
- If verification still fails after 2 fix attempts, stop and report status `failed` with the failing test IDs.
- Your final message lists: files changed, commands run with exit codes, tests added, risks.
- Machine-readable version: `docs/agents/verification-contract.json`.

## Code Review Rules
- Flag float arithmetic on money. Safe path: integer cents.
- Flag API handlers that raise instead of returning the error envelope. Safe path: return `{"error": ...}`.
- Do not comment on formatting; tooling handles it.
"""

CLAUDE_MD = """
@AGENTS.md

## Claude Code only
- Use plan mode for changes that touch more than 3 files or anything in `src/shop/db/migrations/`.
- Run `/codebase-map <package>` before exploring a package you have not read; it runs in a forked context.
- Path-scoped rules in `.claude/rules/` load the `docs/agents/*.md` guides automatically.
"""

write("AGENTS.md", AGENTS_MD)
write("CLAUDE.md", CLAUDE_MD)
print((ROOT / "CLAUDE.md").read_text())

Now simulate what each agent loads at session start. The two resolvers below implement the documented rules that matter here:

- **Claude Code** (default **Project instructions** = `claude-md-or-agents-md`): if any `CLAUDE.md`, `.claude/CLAUDE.md` or `CLAUDE.local.md` exists in the working directory or above, it reads those (expanding `@path` imports, max four hops, never inside code spans or fenced blocks). Otherwise it reads `AGENTS.md`. User-level files load either way.
- **Codex**: `~/.codex/AGENTS.override.md` or `AGENTS.md`, then from the project root down to the working directory at most one of `AGENTS.override.md` / `AGENTS.md` per directory.

In [ ]:
IMPORT_RE = re.compile(r"(?:(?<=\s)|^)@([^\s`]+)")


def find_imports(text: str) -> list[str]:
    """`@path` imports outside fenced code blocks and inline code spans."""
    found, fenced = [], False
    for line in text.splitlines():
        if line.lstrip().startswith("```"):
            fenced = not fenced
            continue
        if not fenced:
            found += IMPORT_RE.findall(re.sub(r"`[^`]*`", "", line))
    return found


def expand_imports(path: Path, home: Path = HOME, depth: int = 0, seen: set | None = None) -> list[Path]:
    """The file plus everything it imports (depth-first, max four hops, each file once)."""
    seen = set() if seen is None else seen
    real = path.resolve()
    if real in seen or not path.exists():
        return []
    seen.add(real)
    files = [path]
    if depth < 4:
        for ref in find_imports(path.read_text()):
            target = (home / ref[2:]) if ref.startswith("~/") else (path.parent / ref)
            files += expand_imports(target, home, depth + 1, seen)
    return files


def dirs_root_to(cwd: Path, root: Path) -> list[Path]:
    rel = cwd.relative_to(root)
    return [root] + [root.joinpath(*rel.parts[: i + 1]) for i in range(len(rel.parts))]


def claude_loads(root: Path, cwd: Path | None = None, home: Path = HOME,
                 mode: str = "claude-md-or-agents-md") -> list[str]:
    cwd = cwd or root
    claude_names = ("CLAUDE.md", ".claude/CLAUDE.md", "CLAUDE.local.md")
    dirs = dirs_root_to(cwd, root)
    has_claude = any((d / n).exists() for d in dirs for n in claude_names)
    names: tuple[str, ...]
    if mode == "claude-md-and-agents-md":
        names = claude_names + ("AGENTS.md", ".claude/AGENTS.md")
    elif mode == "claude-md" or has_claude:
        names = claude_names
    else:
        names = ("AGENTS.md", ".claude/AGENTS.md")
    loaded: list[Path] = []
    seen: set = set()
    for start in [home / ".claude" / "CLAUDE.md"] + [d / n for d in dirs for n in names]:
        loaded += expand_imports(start, home, 0, seen)
    for rule in sorted((root / ".claude" / "rules").glob("**/*.md")):
        if "paths" not in parse_frontmatter(rule.read_text())[0]:
            loaded += expand_imports(rule, home, 0, seen)    # unscoped rules load at launch
    return [str(p.relative_to(home)) if p.is_relative_to(home) else str(p.relative_to(root)) for p in loaded]


def codex_loads(root: Path, cwd: Path | None = None, codex_home: Path = HOME / ".codex") -> list[str]:
    cwd = cwd or root
    chain = []
    for name in ("AGENTS.override.md", "AGENTS.md"):
        p = codex_home / name
        if p.exists() and p.read_text().strip():
            chain.append(f"~/.codex/{name}")
            break
    for d in dirs_root_to(cwd, root):
        for name in ("AGENTS.override.md", "AGENTS.md"):
            p = d / name
            if p.exists() and p.read_text().strip():
                chain.append(str(p.relative_to(root)))
                break
    return chain


def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Split YAML frontmatter (only when `---` is the very first line) from the body."""
    if not text.startswith("---\n"):
        return {}, text
    end = text.find("\n---", 4)
    if end == -1:
        return {}, text
    meta = yaml.safe_load(text[4:end]) or {}
    return (meta if isinstance(meta, dict) else {}), text[end + 4:].lstrip("\n")


print("Claude Code loads:", claude_loads(ROOT))
print("Codex loads:      ", codex_loads(ROOT))

In [ ]:
def bullets(text: str) -> list[str]:
    return [re.sub(r"\s+", " ", l.strip()[2:]).lower() for l in text.splitlines() if l.strip().startswith("- ")]


def check_layering(root: Path = ROOT) -> list[tuple[str, bool, str]]:
    agents = (root / "AGENTS.md").read_text()
    claude = (root / "CLAUDE.md").read_text()
    first = next((l.strip() for l in claude.splitlines() if l.strip()), "")
    dupes = sorted(set(bullets(claude)) & set(bullets(agents)))
    return [
        ("CLAUDE.md starts with @AGENTS.md", first == "@AGENTS.md", first),
        ("no bullet duplicated across files", not dupes, f"{len(dupes)} duplicate(s) {dupes[:2]}"),
        ("AGENTS.md has no @imports (Codex reads them as text)", not find_imports(agents), str(find_imports(agents))),
        ("AGENTS.md under 200 lines", len(agents.splitlines()) < 200, f"{len(agents.splitlines())} lines"),
        ("AGENTS.md well under the 32 KiB Codex cap", len(agents.encode()) < 8 * 1024, f"{len(agents.encode())} bytes"),
        ("Claude Code and Codex both load AGENTS.md", "AGENTS.md" in claude_loads(root) and "AGENTS.md" in codex_loads(root), ""),
    ]


def show_checks(rows: list[tuple[str, bool, str]]) -> bool:
    for name, ok, detail in rows:
        print(f"{'✓' if ok else '✗'} {name}" + (f"  ({detail})" if detail and not ok else ""))
    return all(ok for _, ok, _ in rows)


show_checks(check_layering())

**Pitfall demo.** Make a copy of the repo **without** `CLAUDE.md` (relying on Claude's built-in `AGENTS.md` support), then let one developer add a personal `CLAUDE.local.md`.

In [ ]:
alt = Path(tempfile.mkdtemp(prefix="kestrel-alt-")) / "repo"
shutil.copytree(ROOT, alt, symlinks=True)
(alt / "CLAUDE.md").unlink()
print("no CLAUDE.md:            ", claude_loads(alt))
write("CLAUDE.local.md", "- My sandbox API is http://localhost:9000\n", base=alt)
print("+ personal CLAUDE.local.md:", claude_loads(alt))
print("same, Project instructions = claude-md-and-agents-md (user-level setting):",
      claude_loads(alt, mode="claude-md-and-agents-md"))
write("CLAUDE.local.md", "- My sandbox API is http://localhost:9000\n")   # same personal file in the real design
print("our design + CLAUDE.local.md:", claude_loads(ROOT))
(ROOT / "CLAUDE.local.md").unlink()

> **Observe:** without an explicit import, a personal `CLAUDE.local.md` silently turns `AGENTS.md` off for that developer. The `claude-md-and-agents-md` setting fixes it, but only from user, `--settings` or managed settings, never from the repo. The `@AGENTS.md` import works for everyone.

> **Try it:** add a bullet from `AGENTS.md` to `CLAUDE.md` and rerun `check_layering()`. Then put `@docs/agents/testing.md` into `AGENTS.md` and watch the import check fail.

## 4.2 Path-scoped conventions

Canonical guides live in `docs/agents/` with Claude's `paths:` frontmatter. `.claude/rules/*.md` are symlinks to them, so Claude loads each guide only when it reads, writes or edits a matching file. Codex has no glob-scoped instruction files, so the `AGENTS.md` routing table points it at the same guides. [Theory](README.md#42-path-scoped-conventions)

In [ ]:
GUIDES = {
    "testing": '''
        ---
        paths:
          - "**/test_*.py"
          - "tests/**/*.py"
        ---
        # Testing conventions
        - Use `unittest`; one behavior per test method, named `test_<behavior>`.
        - Build inputs with the helpers in `tests/factories.py`; never hardcode order IDs.
        - Every bug fix adds a regression test that fails before the fix.
        - Before adding a test, list the existing tests for that function and skip scenarios already covered.
    ''',
    "api": '''
        ---
        paths:
          - "src/shop/api/**/*"
        ---
        # API conventions
        - Validate input at the handler boundary; return `{"error": {"code": "invalid_request", ...}}` for bad input.
        - Handlers are pure functions of the payload; no module-level state.
        - Every handler module has a co-located `test_<module>.py`.
    ''',
    "migrations": '''
        ---
        paths: "src/shop/db/migrations/*.sql"
        ---
        # Migration rules
        - Never edit a merged migration; add a new numbered file instead.
        - Every migration has a `-- down:` line that reverses it.
    ''',
}
for name, text in GUIDES.items():
    write(f"docs/agents/{name}.md", text)
    how = link(f".claude/rules/{name}.md", f"docs/agents/{name}.md")
    print(f".claude/rules/{name}.md -> docs/agents/{name}.md ({how})")

In [ ]:
def expand_braces(pattern: str) -> list[str]:
    m = re.search(r"\{([^{}]*)\}", pattern)
    if not m:
        return [pattern]
    return [x for alt in m.group(1).split(",") for x in expand_braces(pattern[: m.start()] + alt + pattern[m.end():])]


def glob_regex(pattern: str) -> str:
    out, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            out, i = out + "(?:.*/)?", i + 3
        elif pattern.startswith("**", i):
            out, i = out + ".*", i + 2
        elif pattern[i] == "*":
            out, i = out + "[^/]*", i + 1
        elif pattern[i] == "?":
            out, i = out + "[^/]", i + 1
        else:
            out, i = out + re.escape(pattern[i]), i + 1
    return out


def glob_match(pattern: str, path: str) -> bool:
    return any(re.fullmatch(glob_regex(p.strip()), path) for p in expand_braces(pattern))


def rule_globs(meta: dict) -> list[str]:
    paths = meta.get("paths", [])
    return [p.strip() for p in paths.split(",")] if isinstance(paths, str) else list(paths)


def rules_for(file_rel: str, root: Path = ROOT) -> list[str]:
    """Path-scoped rules Claude Code would load when it reads/writes/edits file_rel."""
    hits = []
    for rule in sorted((root / ".claude" / "rules").glob("**/*.md")):
        meta, _ = parse_frontmatter(rule.read_text())
        if any(glob_match(g, file_rel) for g in rule_globs(meta)):
            hits.append(rule.name)
    return hits


for f in ["tests/test_pricing.py", "src/shop/api/test_routes.py", "src/shop/api/routes.py",
          "src/shop/pricing.py", "src/shop/db/migrations/0002_add_status.sql", "tests/factories.py"]:
    print(f"{f:45} -> {rules_for(f) or '-'}")

In [ ]:
def check_scoped_rules(root: Path = ROOT) -> list[tuple[str, bool, str]]:
    rows = []
    agents = (root / "AGENTS.md").read_text()
    for guide in sorted((root / "docs" / "agents").glob("*.md")):
        meta, body = parse_frontmatter(guide.read_text())
        rule = root / ".claude" / "rules" / guide.name
        same = rule.exists() and parse_frontmatter(rule.read_text()) == (meta, body)
        rows += [
            (f"{guide.name}: valid paths frontmatter", bool(rule_globs(meta)), str(meta)),
            (f"{guide.name}: .claude/rules copy/symlink matches (no drift)", same,
             "missing" if not rule.exists() else "content differs"),
            (f"{guide.name}: listed in the AGENTS.md routing table", f"`docs/agents/{guide.name}`" in agents, ""),
        ]
    rows += [
        ("testing.md loads for co-located tests", "testing.md" in rules_for("src/shop/api/test_routes.py", root), ""),
        ("testing.md does not load for source files", "testing.md" not in rules_for("src/shop/pricing.py", root), ""),
    ]
    return rows


show_checks(check_scoped_rules())

> **Observe:** `src/shop/api/test_routes.py` triggers **two** rules (testing and API), which is exactly what a directory-only mechanism cannot express. A broken YAML block would make Claude treat the rule as unscoped (always loaded), which is why the check parses every frontmatter block.

> **Try it:** change the migrations guide to `paths: "migrations/*.sql"` and see which check fails. Then replace one `.claude/rules` symlink with an edited copy and rerun `check_scoped_rules()` to see the drift check fire.

## 4.3 Skills both agents can use

`verify-change` is written once in `.agents/skills/` (where Codex looks) with spec-only frontmatter, and `.claude/skills/verify-change` is a symlink to it. `codebase-map` is Claude-only (`context: fork`, `agent: Explore`); its Codex counterpart is a read-only custom agent in `.codex/agents/`. [Theory](README.md#43-skills-both-agents-can-use)

In [ ]:
write(".agents/skills/verify-change/SKILL.md", '''
    ---
    name: verify-change
    description: Verify a code change against the team's verification contract and write the evidence a reviewer needs. Use after editing code and before saying a task is done, or when asked to prove that a change works.
    ---
    # Verify a change

    1. Run `python scripts/verify.py --evidence evidence/verify.json`.
    2. If it fails, read the failing test output and fix the code. Change a test's expectation only when the task
       explicitly changes that behavior. Stop after 2 failed attempts.
    3. Report, in this order: status (`verified` or `failed`), files changed, commands run with exit codes,
       tests added, risks and how to roll back. Follow `docs/agents/verification-contract.json`.
''')
print("verify-change for Claude:", link(".claude/skills/verify-change", ".agents/skills/verify-change"))

write(".claude/skills/codebase-map/SKILL.md", '''
    ---
    name: codebase-map
    description: Map an unfamiliar package (entry points, call paths, tests, risky spots) and return a short summary. Use before changing code you have not read yet.
    argument-hint: "[package-path]"
    context: fork
    agent: Explore
    allowed-tools: Read Grep Glob
    disallowed-tools: Edit Write
    ---
    Map the package at `$ARGUMENTS`:

    1. Find entry points with Grep and Glob; read only the files you need.
    2. Trace the main call paths and list the tests that cover them.
    3. Flag risky spots: money arithmetic, error handling, missing tests.

    Return at most 20 lines. Do not edit anything.
''')

write(".codex/agents/mapper.toml", '''
    name = "mapper"
    description = "Read-only explorer that maps an unfamiliar package and returns a short summary."
    sandbox_mode = "read-only"
    developer_instructions = """
    Map the package you are given: entry points, main call paths, the tests that cover them, risky spots.
    Do not edit files. Return at most 20 lines.
    """
''')
tree(ROOT / ".claude")

In [ ]:
SPEC_KEYS = {"name", "description", "license", "compatibility", "metadata", "allowed-tools"}
CLAUDE_SKILL_KEYS = SPEC_KEYS | {"when_to_use", "argument-hint", "arguments", "disable-model-invocation",
                                 "user-invocable", "disallowed-tools", "model", "effort", "context", "agent",
                                 "background", "hooks", "paths", "shell"}
NAME_RE = re.compile(r"^[a-z0-9]+(-[a-z0-9]+)*$")   # spec: lowercase, digits, single hyphens, 1-64 chars


def tools_of(value: Any) -> set[str]:
    if isinstance(value, list):
        return {str(v).strip() for v in value}
    return {t for t in re.split(r"[\s,]+", str(value or "")) if t}


def validate_skill(skill_dir: Path, portable: bool) -> list[str]:
    """Problems with a skill folder; portable=True means 'must load in any Agent Skills host'."""
    if not (skill_dir / "SKILL.md").exists():
        return [f"{skill_dir.name}/SKILL.md is missing"]
    meta, body = parse_frontmatter((skill_dir / "SKILL.md").read_text())
    name, desc = str(meta.get("name", "")), str(meta.get("description", ""))
    problems = []
    if not (NAME_RE.match(name) and len(name) <= 64):
        problems.append(f"name {name!r} breaks the spec")
    if name != skill_dir.name:
        problems.append(f"name {name!r} must match the folder {skill_dir.name!r}")
    if not 0 < len(desc) <= 1024:
        problems.append("description must be 1-1024 characters")
    allowed = SPEC_KEYS if portable else CLAUDE_SKILL_KEYS
    if extra := set(meta) - allowed:
        problems.append(f"keys not allowed here: {sorted(extra)}")
    if not body.strip():
        problems.append("empty body")
    return problems


def check_skills(root: Path = ROOT) -> list[tuple[str, bool, str]]:
    shared = root / ".agents" / "skills" / "verify-change"
    claude_link = root / ".claude" / "skills" / "verify-change"
    fork_meta, _ = parse_frontmatter((root / ".claude/skills/codebase-map/SKILL.md").read_text())
    mapper = tomllib.loads((root / ".codex/agents/mapper.toml").read_text())
    same = claude_link.exists() and (claude_link.resolve() == shared.resolve()
                                     or (claude_link / "SKILL.md").read_text() == (shared / "SKILL.md").read_text())
    p1, p2 = validate_skill(shared, portable=True), validate_skill(root / ".claude/skills/codebase-map", portable=False)
    return [
        ("verify-change follows the Agent Skills spec (portable)", not p1, "; ".join(p1)),
        (".claude/skills/verify-change is the same skill", same, ""),
        ("codebase-map frontmatter valid for Claude Code", not p2, "; ".join(p2)),
        ("codebase-map runs forked in a read-only agent",
         fork_meta.get("context") == "fork" and fork_meta.get("agent") in {"Explore", "Plan"}, str(fork_meta.get("agent"))),
        ("codebase-map cannot edit", {"Edit", "Write"} <= tools_of(fork_meta.get("disallowed-tools"))
         and not ({"Edit", "Write", "Bash"} & tools_of(fork_meta.get("allowed-tools"))), ""),
        ("Codex mapper agent is complete and read-only",
         {"name", "description", "developer_instructions"} <= set(mapper) and mapper.get("sandbox_mode") == "read-only", ""),
    ]


def codex_skill_dirs(root: Path, cwd: Path | None = None) -> list[str]:
    """Codex scans .agents/skills in every directory from cwd up to the repo root."""
    cwd = cwd or root
    found = []
    for d in reversed(dirs_root_to(cwd, root)):
        found += [str(p.relative_to(root)) for p in sorted((d / ".agents" / "skills").glob("*/SKILL.md"))]
    return found


show_checks(check_skills())
print("Claude Code project skills:", sorted(p.name for p in (ROOT / ".claude/skills").iterdir()))
print("Codex repo skills (cwd=src/shop):", codex_skill_dirs(ROOT, ROOT / "src" / "shop"))

> **Observe:** Codex finds the shared skill even when started in `src/shop/`, because it scans `.agents/skills` from the working directory **up** to the repo root. In the opposite direction (instruction files), it never looks **down**.

> **Try it:** add `context: fork` to the shared skill and rerun the checks. The portable check fails: that key is a Claude Code extension, and a host that validates the spec strictly (such as a claude.ai upload) rejects unknown keys.

## 4.4 MCP servers: team and personal

The team tracker server is committed for both tools with credentials from the environment. Each developer's experimental server stays in their own home directory. The resolvers below reproduce the scope rules and Claude Code's `${VAR}` expansion, including the credential variables it deliberately reads as empty for remote servers. [Theory](README.md#44-mcp-servers-team-and-personal)

In [ ]:
write(".mcp.json", json.dumps({
    "mcpServers": {
        "tracker": {
            "type": "http",
            "url": "${TRACKER_MCP_URL:-https://tracker.kestrel.example/mcp}",
            "headers": {"Authorization": "Bearer ${TRACKER_TOKEN}"},
        }
    }
}, indent=2) + "\n")

write(".codex/config.toml", '''
    # Team defaults for Codex. Codex loads this file only when the project is trusted.
    approval_policy = "on-request"
    sandbox_mode = "workspace-write"

    [sandbox_workspace_write]
    network_access = false

    [mcp_servers.tracker]
    url = "https://tracker.kestrel.example/mcp"
    bearer_token_env_var = "TRACKER_TOKEN"
''')

# Personal config in the FAKE home directory (never in git)
(HOME / ".claude.json").write_text(json.dumps({
    "projects": {str(ROOT): {"mcpServers": {          # local scope: this project, this developer only
        "scratch-notes": {"type": "stdio", "command": "python", "args": ["-m", "notes_mcp"]}}}},
}, indent=2))
write(".codex/config.toml", '''
    model = "gpt-6-luna"

    [mcp_servers.scratch_notes]
    command = "python"
    args = ["-m", "notes_mcp"]
''', base=HOME)
print("written: .mcp.json, .codex/config.toml, ~/.claude.json, ~/.codex/config.toml (fake home)")

In [ ]:
# Illustrative subset of the variables Claude Code reads as empty in a REMOTE server's url/headers.
BLANKED_FOR_REMOTE = {"ANTHROPIC_API_KEY", "ANTHROPIC_AUTH_TOKEN", "AWS_BEARER_TOKEN_BEDROCK", "HTTPS_PROXY", "NPM_TOKEN"}
VAR_RE = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)(?::-([^}]*))?\}")


def expand_value(value: Any, env: dict, blank_credentials: bool) -> tuple[Any, list[str]]:
    """Expand ${VAR} / ${VAR:-default} recursively; returns (value, warnings)."""
    if isinstance(value, dict):
        pairs = {k: expand_value(v, env, blank_credentials) for k, v in value.items()}
        return {k: p[0] for k, p in pairs.items()}, [w for p in pairs.values() for w in p[1]]
    if isinstance(value, list):
        pairs = [expand_value(v, env, blank_credentials) for v in value]
        return [p[0] for p in pairs], [w for p in pairs for w in p[1]]
    if not isinstance(value, str):
        return value, []
    warnings: list[str] = []

    def sub(m: re.Match) -> str:
        name, default = m.group(1), m.group(2)
        if blank_credentials and name in BLANKED_FOR_REMOTE:
            warnings.append(f"{name} is never expanded toward a remote server (sent empty)")
            return ""
        if name in env:
            return env[name]
        if default is not None:
            return default
        warnings.append(f"{name} is not set; left unexpanded")
        return m.group(0)

    return VAR_RE.sub(sub, value), warnings


def expand_server(cfg: dict, env: dict) -> tuple[dict, list[str]]:
    """Claude Code expands command, args, env, url and headers; credential variables read as empty
    only in a REMOTE server's url and headers."""
    remote = cfg.get("type") in {"http", "sse", "streamable-http"}
    out, warnings = {}, []
    for key, value in cfg.items():
        out[key], w = expand_value(value, env, blank_credentials=remote and key in {"url", "headers"})
        warnings += w
    return out, warnings


def claude_mcp_view(root: Path, home: Path, env: dict) -> dict[str, dict]:
    """Servers one Claude Code session sees: local > project > user (whole entry from the winner)."""
    user_cfg = json.loads((home / ".claude.json").read_text()) if (home / ".claude.json").exists() else {}
    scopes = [
        ("local", user_cfg.get("projects", {}).get(str(root), {}).get("mcpServers", {})),
        ("project", json.loads((root / ".mcp.json").read_text()).get("mcpServers", {}) if (root / ".mcp.json").exists() else {}),
        ("user", user_cfg.get("mcpServers", {})),
    ]
    view: dict[str, dict] = {}
    for scope, servers in scopes:
        for name, cfg in servers.items():
            if name not in view:
                expanded, warnings = expand_server(cfg, env)
                view[name] = {"scope": scope, "config": expanded, "warnings": warnings}
    return view


def codex_mcp_view(root: Path, codex_home: Path, env: dict, trusted: bool) -> dict[str, dict]:
    """Servers Codex sees: user config, then the project layer (closest wins) if the project is trusted."""
    layers = [("user", codex_home / "config.toml")] + ([("project", root / ".codex" / "config.toml")] if trusted else [])
    view: dict[str, dict] = {}
    for scope, path in layers:
        for name, cfg in (tomllib.loads(path.read_text()).get("mcp_servers", {}) if path.exists() else {}).items():
            token_var = cfg.get("bearer_token_env_var")
            view[name] = {"scope": scope, "config": cfg,
                          "warnings": [f"{token_var} is not set"] if token_var and token_var not in env else []}
    return view


ENV = {"TRACKER_TOKEN": "tok_dev_123"}  # what a developer's shell would export
for name, s in claude_mcp_view(ROOT, HOME, ENV).items():
    print("claude", f"{name:14} {s['scope']:8}", s["config"].get("url") or s["config"].get("command"), s["warnings"] or "")
for trusted in (True, False):
    print(f"codex (trusted={trusted}):", {n: s["scope"] for n, s in codex_mcp_view(ROOT, HOME / ".codex", ENV, trusted).items()})

In [ ]:
SECRET_PATTERNS = [
    re.compile(r"sk-[A-Za-z0-9_-]{16,}"),                       # API-key-shaped strings
    re.compile(r"gh[pousr]_[A-Za-z0-9]{20,}"),                  # GitHub tokens
    re.compile(r"Bearer\s+(?!\$\{)[A-Za-z0-9._-]{12,}"),       # literal bearer tokens
]


def scan_for_secrets(root: Path = ROOT) -> list[str]:
    hits = []
    for p in root.rglob("*"):
        if p.is_file() and not p.is_symlink() and "__pycache__" not in p.parts and p.suffix in {".json", ".toml", ".md", ".yml", ".yaml", ".py", ".rules"}:
            for pat in SECRET_PATTERNS:
                if pat.search(p.read_text(errors="ignore")):
                    hits.append(f"{p.relative_to(root)}: {pat.pattern}")
    return hits


def check_mcp(root: Path = ROOT, home: Path = HOME) -> list[tuple[str, bool, str]]:
    env = {"TRACKER_TOKEN": "x"}
    cview, xview = claude_mcp_view(root, home, env), codex_mcp_view(root, home / ".codex", env, trusted=True)
    team_header = json.loads((root / ".mcp.json").read_text())["mcpServers"]["tracker"]["headers"]["Authorization"]
    blanked = [v for v in VAR_RE.findall(json.dumps(json.loads((root / ".mcp.json").read_text()))) if v[0] in BLANKED_FOR_REMOTE]
    return [
        ("Claude team server takes its token from the environment", "${TRACKER_TOKEN}" in team_header, team_header),
        ("no credential variable that Claude Code blanks for remote servers", not blanked, str(blanked)),
        ("Codex team server uses bearer_token_env_var",
         xview.get("tracker", {}).get("config", {}).get("bearer_token_env_var") == "TRACKER_TOKEN", ""),
        ("Claude: personal + team servers in one session", {"tracker", "scratch-notes"} <= set(cview), str(set(cview))),
        ("Codex: personal + team servers in one session", {"tracker", "scratch_notes"} <= set(xview), str(set(xview))),
        ("no secrets committed", not scan_for_secrets(root), "; ".join(scan_for_secrets(root))),
    ]


show_checks(check_mcp())

bad = json.loads((ROOT / ".mcp.json").read_text())
bad["mcpServers"]["tracker"]["headers"]["Authorization"] = "Bearer ${ANTHROPIC_API_KEY}"
print("\nmis-configured header ->", expand_server(bad["mcpServers"]["tracker"], {"ANTHROPIC_API_KEY": "sk-ant-example"}))

> **Observe:** the same session shows the committed `tracker` server and the personal `scratch-notes` server, from different scopes. For Codex, the team server disappears when the project is not trusted: project `.codex/` layers are skipped entirely.

> **Try it:** add a user-scope (`"mcpServers"` at the top level of `~/.claude.json`) server also called `tracker` and check which definition wins.

## 4.5 Permissions, sandbox and approvals

Instruction files are guidance; these files are enforcement. Claude Code evaluates `deny` → `ask` → `allow` (first match wins). Codex `.rules` files apply the most restrictive matching decision (`forbidden` > `prompt` > `allow`) and self-test their `match` / `not_match` examples on load. [Theory](README.md#45-permissions-sandbox-and-approvals)

In [ ]:
write(".claude/settings.json", json.dumps({
    "$schema": "https://json.schemastore.org/claude-code-settings.json",
    "permissions": {
        "allow": ["Bash(python scripts/verify.py *)", "Bash(git diff *)", "Bash(git status *)"],
        "ask": ["Bash(git push *)"],
        "deny": ["Read(./.env)", "Read(./.env.*)", "Bash(terraform apply *)",
                 # force-push in the spellings Claude usually writes: flag first, or after remote/branch
                 "Bash(git push --force *)", "Bash(git push -f *)", "Bash(git push * --force*)", "Bash(git push * -f*)"],
    },
    "sandbox": {"enabled": True, "network": {"allowedDomains": ["pypi.org", "files.pythonhosted.org"]}},
}, indent=2) + "\n")

write(".codex/rules/team.rules", '''
    # Codex command rules for kestrel-shop (Starlark; experimental feature). Loaded only in trusted projects.
    prefix_rule(
        pattern = ["git", "push"],
        decision = "prompt",
        justification = "Pushing needs a human. Open a pull request instead.",
        match = ["git push origin feature/discounts"],
        not_match = ["git status"],
    )

    prefix_rule(
        pattern = ["git", "push", ["--force", "-f"]],
        decision = "forbidden",
        justification = "Force-push rewrites shared history. Push a new commit instead.",
        match = ["git push --force origin main", "git push -f"],
        not_match = ["git push origin main"],
    )

    prefix_rule(
        pattern = ["terraform", "apply"],
        decision = "forbidden",
        justification = "Infrastructure changes go through the deploy pipeline.",
        match = ["terraform apply -auto-approve"],
    )
''')
print((ROOT / ".claude/settings.json").read_text())

In [ ]:
RULE_RE = re.compile(r"^(\w+)(?:\((.*)\))?$")


def bash_rule_matches(spec: str, command: str) -> bool:
    """Claude Code Bash rule semantics used here: `*` is a wildcard; a trailing ` *` that is the only
    wildcard also matches the bare command (`Bash(git push *)` matches `git push`, not `git pushx`)."""
    command = " ".join(command.split())
    if spec.endswith(" *") and spec.count("*") == 1:
        prefix = spec[:-2]
        return command == prefix or command.startswith(prefix + " ")
    return re.fullmatch(".*".join(re.escape(p) for p in spec.split("*")), command) is not None


def path_rule_matches(spec: str, file_rel: str) -> bool:
    spec = spec[2:] if spec.startswith("./") else spec.lstrip("/")
    return glob_match(spec, file_rel)


def claude_decide(tool: str, arg: str, settings: dict) -> str:
    """'deny' | 'ask' | 'allow' | 'default' (no rule matched: the permission mode decides)."""
    perms = settings.get("permissions", {})
    for decision in ("deny", "ask", "allow"):
        for rule in perms.get(decision, []):
            m = RULE_RE.match(rule)
            if not m or m.group(1) != tool:
                continue
            spec = m.group(2)
            if spec is None or (tool == "Bash" and bash_rule_matches(spec, arg)) or \
               (tool in {"Read", "Edit"} and path_rule_matches(spec, arg)):
                return decision
    return "default"


def load_codex_rules(text: str) -> list[dict]:
    """Parse prefix_rule(...) calls without executing anything (the file is Starlark, Python-compatible here)."""
    rules = []
    for node in ast.walk(ast.parse(text)):
        if isinstance(node, ast.Call) and getattr(node.func, "id", "") == "prefix_rule":
            kw = {k.arg: ast.literal_eval(k.value) for k in node.keywords}
            kw.setdefault("decision", "allow")
            rules.append(kw)
    return rules


def prefix_matches(pattern: list, argv: list[str]) -> bool:
    if len(argv) < len(pattern):
        return False
    return all(a in (p if isinstance(p, list) else [p]) for p, a in zip(pattern, argv))


SAFE_WORD = re.compile(r"^[A-Za-z0-9_./:=@%+-]+$")


def split_shell(argv: list[str]) -> list[list[str]]:
    """Split `bash -lc "a && b"` into commands when it is a plain chain (no $, *, redirects, quotes...)."""
    if len(argv) == 3 and argv[0] in {"bash", "sh", "zsh"} and argv[1] in {"-lc", "-c"}:
        parts = re.split(r"\s*(?:&&|\|\||;|\|)\s*", argv[2].strip())
        words = [p.split() for p in parts]
        if all(w and all(SAFE_WORD.match(x) for x in w) for w in words):
            return words
    return [argv]


ORDER = {"allow": 0, "prompt": 1, "forbidden": 2}


def codex_decide(argv: list[str], rules: list[dict]) -> str | None:
    """Most restrictive matching decision across every command in the (split) invocation; None = no rule."""
    decisions = [r["decision"] for cmd in split_shell(argv) for r in rules if prefix_matches(r["pattern"], cmd)]
    return max(decisions, key=ORDER.__getitem__) if decisions else None


def self_test_rules(rules: list[dict]) -> list[str]:
    errors = []
    for r in rules:
        errors += [f"{r['pattern']} should match {ex!r}" for ex in r.get("match", []) if not prefix_matches(r["pattern"], shlex.split(ex))]
        errors += [f"{r['pattern']} should NOT match {ex!r}" for ex in r.get("not_match", []) if prefix_matches(r["pattern"], shlex.split(ex))]
    return errors


SETTINGS = json.loads((ROOT / ".claude/settings.json").read_text())
CODEX_RULES = load_codex_rules((ROOT / ".codex/rules/team.rules").read_text())
print("Codex rule self-test:", self_test_rules(CODEX_RULES) or "all examples pass")
for cmd in ["python scripts/verify.py --evidence evidence/verify.json", "git push", "git push origin main",
            "git push --force origin main", "git push -f", "git push origin main --force", "terraform apply", "rm -rf build"]:
    print(f"{cmd:55} claude={claude_decide('Bash', cmd, SETTINGS):8} codex={codex_decide(shlex.split(cmd), CODEX_RULES)}")
print("Read .env ->", claude_decide("Read", ".env", SETTINGS), "| Read .env.production ->", claude_decide("Read", ".env.production", SETTINGS))
print("smuggled chain ->", codex_decide(["bash", "-lc", "git add . && git push -f"], CODEX_RULES))

In [ ]:
GUARDRAIL_MATRIX = [
    # (context, Claude Code, Codex)
    ("laptop default", "project allow/ask/deny + sandbox.enabled", "workspace-write + on-request, network off, team.rules"),
    ("exploring / reviewing", "plan mode (--permission-mode plan)", "read-only sandbox, /plan"),
    ("CI review", "-p --tools Read,Grep,Glob --disallowedTools mcp__* --permission-mode dontAsk", "exec --sandbox read-only"),
    ("CI test generation", "-p --permission-mode acceptEdits + allowedTools", "exec --sandbox workspace-write"),
    ("never, on any machine", "managed-settings deny rules", "requirements.toml forbidden rules"),
]
for row in GUARDRAIL_MATRIX:
    print(f"{row[0]:24} | {row[1]:76} | {row[2]}")


FORCE_PUSHES = ("git push --force origin main", "git push -f", "git push -f origin main",
                "git push origin main --force", "git push origin feature/x -f")


def check_guardrails(root: Path = ROOT) -> list[tuple[str, bool, str]]:
    s = json.loads((root / ".claude/settings.json").read_text())
    rules = load_codex_rules((root / ".codex/rules/team.rules").read_text())
    cfg = tomllib.loads((root / ".codex/config.toml").read_text())
    return [
        ("Claude denies .env reads", claude_decide("Read", ".env", s) == "deny", ""),
        ("Claude denies force-push (flag first or last, --force or -f) and terraform apply",
         {claude_decide("Bash", c, s) for c in FORCE_PUSHES + ("terraform apply",)} == {"deny"},
         str({c: claude_decide("Bash", c, s) for c in FORCE_PUSHES})),
        ("Claude asks before any other push", claude_decide("Bash", "git push origin main", s) == "ask", ""),
        ("Claude runs the verify script without prompting", claude_decide("Bash", "python scripts/verify.py", s) == "allow", ""),
        ("Claude Bash sandbox enabled", s.get("sandbox", {}).get("enabled") is True, ""),
        ("Codex defaults: workspace-write + on-request, network off",
         cfg.get("sandbox_mode") == "workspace-write" and cfg.get("approval_policy") == "on-request"
         and cfg.get("sandbox_workspace_write", {}).get("network_access") is False, ""),
        ("Codex forbids force-push with the flag right after `push`, and terraform apply",
         {codex_decide(shlex.split(c), rules) for c in ("git push -f", "git push --force origin main", "terraform apply")} == {"forbidden"}, ""),
        ("Codex: a trailing --force still needs a human (prefix rules cannot see it; branch protection is the backstop)",
         codex_decide(shlex.split("git push origin main --force"), rules) == "prompt", ""),
        ("Codex rules pass their own examples", not self_test_rules(rules), "; ".join(self_test_rules(rules))),
    ]


show_checks(check_guardrails())

> **Observe:** `git push --force origin main` matches **both** Codex rules (`prompt` and `forbidden`) and the most restrictive one wins. In Claude Code the `deny` list is checked first, so the broader `ask` rule for `git push *` never gets a chance. The smuggled `git add . && git push -f` is split and still forbidden.

> **Observe (the limits):** both rule engines match command **text**, so test the spellings an agent actually writes. Claude's deny list covers the flag first or last (`git push origin main --force`); Codex prefix rules compare arguments position by position and have no wildcards, so the trailing `--force` form only reaches the `git push` rule and still **prompts** a human. Neither engine sees `git -C . push -f`, `git push origin +main` (a force push through a `+` refspec) or a push inside `sh -c '...'` (Claude's permissions page lists such forms under "what a Bash rule doesn't match"). The controls that hold whatever the spelling are the sandbox with network off, a PreToolUse hook that parses the command if you need more, and **server-side branch protection** that rejects force pushes to `main` for every client.

> **Try it:** add `"Bash(git push *)"` to Claude's `allow` list. Does that unblock force-push? (It does not: `deny` is evaluated before `allow`, whatever the specificity.)

## 4.6 Plan mode vs direct execution

Exercise 2 compares the two modes on three tasks. The deciding factors are **scope** (how many files, how much unknown code) and **ambiguity** (how many valid designs). [Theory](README.md#46-plan-mode-vs-direct-execution)

In [ ]:
TASKS = [
    {"id": "bugfix", "title": "Unknown discount code raises KeyError (stack trace points at pricing.py)",
     "files": 1, "valid_designs": 1, "clear_cause": True},
    {"id": "migration", "title": "Replace datetime.utcnow() with timezone-aware datetime.now(UTC) everywhere",
     "files": 45, "valid_designs": 1, "clear_cause": True},
    {"id": "feature", "title": "Order-status notifications: webhook, polling export or message queue?",
     "files": 6, "valid_designs": 3, "clear_cause": False},
]


def choose_mode(task: dict) -> tuple[str, str]:
    if task["valid_designs"] > 1:
        return "plan", "several valid designs: a human should pick one before code is written"
    if task["files"] > 3:
        return "plan", "many files: map the change and its order first, then execute the plan"
    if not task["clear_cause"]:
        return "plan", "unclear cause: investigate before editing"
    return "direct", "small, well-understood change"


ENTER_PLAN = {"Claude Code": "Shift+Tab, /plan <prompt>, or claude --permission-mode plan",
              "Codex": "/plan <prompt> in the TUI (read-only sandbox for exploration)"}
for t in TASKS:
    mode, why = choose_mode(t)
    print(f"[{mode:6}] {t['title']}\n          {why}")
print("\nEntering plan mode:", json.dumps(ENTER_PLAN, indent=1))

In [ ]:
def git_init(root: Path = ROOT) -> None:
    """Live runs only: both CLIs expect a Git repository."""
    if (root / ".git").exists():
        return
    for args in (["init", "-q"], ["add", "-A"], ["-c", "user.name=capstone", "-c", "user.email=capstone@example.com",
                                                 "commit", "-qm", "baseline"]):
        subprocess.run(["git", *args], cwd=root, check=True, capture_output=True)


plan_prompt = ("Plan (do not edit) how to replace datetime.utcnow() with datetime.now(UTC) across this repo. "
               "List files, order of changes, and how to verify.")
claude_plan_argv = ["claude", "-p", plan_prompt, "--permission-mode", "plan", "--output-format", "json",
                    "--max-turns", "6", "--max-budget-usd", MAX_BUDGET_USD, "--model", CLAUDE_MODEL]
codex_plan_argv = ["codex", "exec", "--sandbox", "read-only", "--ephemeral", "--model", OPENAI_MODEL, plan_prompt]

if RUN_CLAUDE and CLAUDE_BIN:
    git_init()
    out = subprocess.run(claude_plan_argv, cwd=ROOT, capture_output=True, text=True, timeout=600)
    print(json.loads(out.stdout).get("result", "")[:1500] if out.stdout.strip() else out.stderr[-800:])
else:
    print("[offline] $", shlex.join(claude_plan_argv)[:220], "...")
if RUN_CODEX and CODEX_BIN:
    git_init()
    out = subprocess.run(codex_plan_argv, cwd=ROOT, capture_output=True, text=True, timeout=600)
    print(out.stdout[-1500:] or out.stderr[-800:])
else:
    print("[offline] $", shlex.join(codex_plan_argv)[:220], "...")

> **Observe:** the migration has only one valid design, yet it still goes to plan mode because of its **size**: the plan becomes the checklist you then execute directly. The notification feature goes to plan mode because of **ambiguity**.

> **Try it:** add a fourth task, "rename `apply_discount` to `discounted_price` across 2 files with a clear spec", and predict the mode before running `choose_mode`.

## 4.7 CI review with structured output

One schema file serves both CLIs: Claude Code gets it inline with `--json-schema` and returns `structured_output`; Codex reads it with `--output-schema` and returns the final message as a JSON string. Write it in the strict subset (every property required, `additionalProperties: false`). [Theory](README.md#47-ci-review-with-structured-output)

In [ ]:
REVIEW_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["summary", "findings"],
    "properties": {
        "summary": {"type": "string"},
        "findings": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["file", "line", "severity", "category", "issue", "evidence", "suggested_fix"],
                "properties": {
                    "file": {"type": "string"},
                    "line": {"type": "integer"},
                    "severity": {"type": "string", "enum": ["critical", "high", "medium", "low"]},
                    "category": {"type": "string", "enum": ["correctness", "security", "tests", "conventions", "performance"]},
                    "issue": {"type": "string"},
                    "evidence": {"type": "string"},
                    "suggested_fix": {"type": "string"},
                },
            },
        },
    },
}
REVIEW_PROMPT = """
You are an independent code reviewer for kestrel-shop. Review only the lines this pull request adds or changes
(run `git diff origin/main...HEAD` if the diff is not on stdin).

- Apply the "Code Review Rules" and conventions in AGENTS.md and docs/agents/.
- Report only concrete problems a maintainer would fix: correctness, security, missing tests, team conventions.
- `line` is the line number in the NEW file. `evidence` is the exact text of that line, copied verbatim.
- If previous findings are provided, report only issues that are new or still unresolved.
- No style or formatting comments. If there is nothing to report, return an empty findings list.
"""
write(".github/agent-review/review.schema.json", json.dumps(REVIEW_SCHEMA, indent=2) + "\n")
write(".github/agent-review/review-prompt.md", REVIEW_PROMPT)


def strict_problems(schema: dict, where: str = "$") -> list[str]:
    """Where a schema leaves the strict subset that both CLIs accept without surprises."""
    problems = []
    if schema.get("type") == "object":
        props = schema.get("properties", {})
        if schema.get("additionalProperties") is not False:
            problems.append(f"{where}: additionalProperties must be false")
        if set(schema.get("required", [])) != set(props):
            problems.append(f"{where}: every property must be required (use a null type for optional values)")
        for k, sub in props.items():
            problems += strict_problems(sub, f"{where}.{k}")
    if schema.get("type") == "array":
        problems += strict_problems(schema.get("items", {}), f"{where}[]")
    return problems


TYPES = {"object": dict, "array": list, "string": str, "integer": int, "number": (int, float), "boolean": bool, "null": type(None)}


def validate(instance: Any, schema: dict, where: str = "$") -> list[str]:
    """Tiny JSON Schema check for the keywords used in this notebook (type, enum, required, properties, items)."""
    types = schema.get("type")
    types = types if isinstance(types, list) else [types] if types else []
    if types and not any(isinstance(instance, TYPES[t]) and not (t in {"integer", "number"} and isinstance(instance, bool)) for t in types):
        return [f"{where}: expected {types}, got {type(instance).__name__}"]
    errors = []
    if "enum" in schema and instance not in schema["enum"]:
        errors.append(f"{where}: {instance!r} not in {schema['enum']}")
    if isinstance(instance, dict):
        errors += [f"{where}: missing {k}" for k in schema.get("required", []) if k not in instance]
        if schema.get("additionalProperties") is False:
            errors += [f"{where}: unexpected {k}" for k in instance if k not in schema.get("properties", {})]
        for k, sub in schema.get("properties", {}).items():
            if k in instance:
                errors += validate(instance[k], sub, f"{where}.{k}")
    if isinstance(instance, list) and "items" in schema:
        for i, item in enumerate(instance):
            errors += validate(item, schema["items"], f"{where}[{i}]")
    return errors


print("strict subset problems:", strict_problems(REVIEW_SCHEMA) or "none")

In [ ]:
def claude_review_argv(prompt: str, schema: dict, bare: bool = False, model: str = CLAUDE_MODEL) -> list[str]:
    argv = ["claude", "-p", prompt,                      # prompt right after -p: list flags would swallow it
            "--output-format", "json", "--json-schema", json.dumps(schema, separators=(",", ":")),
            # --tools REMOVES every other built-in tool (no Edit, Write or Bash at all); --allowedTools would only
            # pre-approve. MCP tools are outside --tools, so deny them; dontAsk denies anything that would prompt.
            "--tools", "Read,Grep,Glob", "--disallowedTools", "mcp__*", "--permission-mode", "dontAsk",
            "--max-turns", "8", "--max-budget-usd", MAX_BUDGET_USD, "--model", model]
    if bare:  # skip hooks, MCP, CLAUDE.md auto-discovery; pass the shared instructions explicitly
        argv += ["--bare", "--append-system-prompt-file", "AGENTS.md"]
    return argv


def codex_review_argv(prompt: str, schema_path: str = ".github/agent-review/review.schema.json",
                      model: str = OPENAI_MODEL) -> list[str]:
    return ["codex", "exec", "--json", "--ephemeral", "--sandbox", "read-only",
            "--model", model, "--output-schema", schema_path, prompt]


print("$", shlex.join(claude_review_argv("<review prompt>", REVIEW_SCHEMA, bare=True))[:300], "...")
print("$", shlex.join(codex_review_argv("<review prompt>")))

Before the outputs make sense, we need a pull request. The PR below adds a `bulk_quote` handler with two real problems (it raises on bad input and does money math in `float`).

In [ ]:
OLD_ROUTES = (ROOT / "src/shop/api/routes.py").read_text()
NEW_ROUTES = OLD_ROUTES + textwrap.dedent('''

    def bulk_quote(payload: dict) -> dict:
        items = payload["items"]
        total = sum(item["price_cents"] * 1.0 for item in items)
        return {"total_cents": apply_discount(int(total), payload.get("code"))}
''')


def unified(old: str, new: str, path: str) -> str:
    return "".join(difflib.unified_diff(old.splitlines(True), new.splitlines(True), f"a/{path}", f"b/{path}"))


PR_DIFF = unified(OLD_ROUTES, NEW_ROUTES, "src/shop/api/routes.py")
print(PR_DIFF)


def line_of(text: str, needle: str) -> int:
    return next(i for i, l in enumerate(text.splitlines(), 1) if needle in l)

In [ ]:
def sample_finding(file, line, severity, category, issue, evidence, fix) -> dict:
    return {"file": file, "line": line, "severity": severity, "category": category,
            "issue": issue, "evidence": evidence, "suggested_fix": fix}


R = "src/shop/api/routes.py"
L_ITEMS, L_TOTAL, L_DEF = (line_of(NEW_ROUTES, s) for s in ('payload["items"]', "* 1.0", "def bulk_quote"))
F_RAISE = sample_finding(R, L_ITEMS, "high", "conventions", "bulk_quote raises KeyError on a payload without items; handlers must return the error envelope.",
                         'items = payload["items"]', 'Validate items and return {"error": {"code": "invalid_request", ...}}.')
F_FLOAT = sample_finding(R, L_TOTAL, "high", "correctness", "Money is summed as float; large carts can lose cents.",
                         'total = sum(item["price_cents"] * 1.0 for item in items)', 'total = sum(item["price_cents"] for item in items)')
F_GHOST = sample_finding(R, 3, "critical", "security", "User input reaches eval().", "result = eval(payload)", "Remove eval.")
F_OUTSIDE = sample_finding("src/shop/pricing.py", line_of((ROOT / "src/shop/pricing.py").read_text(), "DISCOUNT_PERCENT[code"),
                           "medium", "correctness", "Unknown discount codes raise KeyError.",
                           "percent = DISCOUNT_PERCENT[code.upper()]", "Use DISCOUNT_PERCENT.get(code.upper(), 0).")
F_NOTEST = sample_finding(R, L_DEF, "medium", "tests", "bulk_quote has no co-located test.",
                          "def bulk_quote(payload: dict) -> dict:", "Add test_bulk_quote_* cases to src/shop/api/test_routes.py.")

# SAMPLE outputs, shaped like the real CLIs' output
CLAUDE_SAMPLE_STDOUT = json.dumps({
    "type": "result", "subtype": "success", "is_error": False, "num_turns": 4, "duration_ms": 18250,
    "result": "", "session_id": "sample-claude-session", "total_cost_usd": 0.041,
    "structured_output": {"summary": "Two convention/correctness problems in bulk_quote.",
                          "findings": [F_RAISE, F_FLOAT, F_GHOST, F_OUTSIDE]},
})
CODEX_SAMPLE_JSONL = "\n".join(json.dumps(e) for e in [
    {"type": "thread.started", "thread_id": "sample-codex-thread"},
    {"type": "turn.started"},
    {"type": "item.completed", "item": {"id": "item_0", "type": "command_execution",
                                        "command": "bash -lc 'git diff origin/main...HEAD'", "exit_code": 0, "status": "completed"}},
    {"type": "item.completed", "item": {"id": "item_1", "type": "agent_message", "text": json.dumps(
        {"summary": "Float money math and a missing test.", "findings": [F_FLOAT, F_NOTEST]})}},
    {"type": "turn.completed", "usage": {"input_tokens": 21000, "cached_input_tokens": 18000, "output_tokens": 410}},
])
print(CODEX_SAMPLE_JSONL[:400], "...")

In [ ]:
class AgentRunError(RuntimeError):
    """The agent run did not finish cleanly; never treat it as 'no findings'."""


def findings_from_claude(stdout: str) -> dict:
    out = json.loads(stdout)
    if out.get("is_error") or out.get("subtype") != "success" or "structured_output" not in out:
        raise AgentRunError(f"claude run failed: subtype={out.get('subtype')}")
    return out["structured_output"]


def findings_from_codex(jsonl: str) -> dict:
    final, completed = None, False
    for line in jsonl.splitlines():
        try:
            ev = json.loads(line)
        except json.JSONDecodeError:
            continue
        if ev.get("type") == "item.completed" and ev.get("item", {}).get("type") == "agent_message":
            final = ev["item"]["text"]
        elif ev.get("type") == "turn.completed":
            completed = True
        elif ev.get("type") in {"turn.failed", "error"}:
            raise AgentRunError(f"codex run failed: {ev}")
    if not completed or final is None:
        raise AgentRunError("codex run did not complete")
    return json.loads(final)


def collect(source: str, payload: dict) -> list[dict]:
    errors = validate(payload, REVIEW_SCHEMA)
    if errors:
        raise AgentRunError(f"{source} output breaks the schema: {errors[:3]}")
    return [{**f, "source": source} for f in payload["findings"]]


ALL_FINDINGS = collect("claude", findings_from_claude(CLAUDE_SAMPLE_STDOUT)) + collect("codex", findings_from_codex(CODEX_SAMPLE_JSONL))
for f in ALL_FINDINGS:
    print(f"[{f['source']:6}] {f['file']}:{f['line']:<3} {f['severity']:8} {f['category']:12} {f['issue'][:60]}")
for bad_run in ('{"type":"result","subtype":"error_max_turns","is_error":true}',):
    try:
        findings_from_claude(bad_run)
    except AgentRunError as e:
        print("incomplete run ->", e)

Now the GitHub workflow. It is generated from the same schema and prompt files, then audited: key-holding jobs must not be able to write, the writing job must not hold model keys, and the schema inlined into `claude_args` must equal the file Codex reads.

> **Checked offline only.** This notebook parses and audits the workflow, and tests `post_review.py` against a fake GitHub API. Neither has run on GitHub. Before you rely on them, open one test PR and confirm three things: the Claude job passes the action's write-access check with the read-only `GITHUB_TOKEN`, both `findings` outputs arrive in `post-comments`, and the review posts once.

In [ ]:
def build_workflow(schema: dict) -> str:
    inline = json.dumps(schema, separators=(",", ":"))
    return textwrap.dedent(f"""\
    name: Agent review
    on:
      pull_request:
        types: [opened, synchronize, reopened, ready_for_review]

    permissions: {{}}

    jobs:
      claude-review:
        if: github.event.pull_request.draft == false
        runs-on: ubuntu-latest
        permissions:          # no id-token: write, so the action cannot fetch a Claude GitHub App token
          contents: read
          pull-requests: read
        outputs:
          findings: ${{{{ steps.claude.outputs.structured_output }}}}
        steps:
          - uses: actions/checkout@v6
            with:
              fetch-depth: 0
              persist-credentials: false
          - id: prompt
            run: |
              {{
                echo 'text<<PROMPT_EOF'
                cat .github/agent-review/review-prompt.md
                echo 'PROMPT_EOF'
              }} >> "$GITHUB_OUTPUT"
          - id: claude
            uses: anthropics/claude-code-action@v1
            with:
              anthropic_api_key: ${{{{ secrets.ANTHROPIC_API_KEY }}}}
              # The workflow token is scoped by this job's permissions block (read-only).
              # Without it, the action authenticates as the Claude GitHub App, which this block does not scope.
              github_token: ${{{{ secrets.GITHUB_TOKEN }}}}
              prompt: ${{{{ steps.prompt.outputs.text }}}}
              claude_args: >-
                --max-turns 8
                --allowedTools "Read,Grep,Glob,Bash(git diff *)"
                --json-schema '{inline}'

      codex-review:
        if: github.event.pull_request.draft == false
        runs-on: ubuntu-latest
        permissions:
          contents: read
        outputs:
          findings: ${{{{ steps.codex.outputs.final-message }}}}
        steps:
          - uses: actions/checkout@v6
            with:
              ref: refs/pull/${{{{ github.event.pull_request.number }}}}/merge
              fetch-depth: 0
              persist-credentials: false
          - id: codex
            uses: openai/codex-action@v1
            with:
              openai-api-key: ${{{{ secrets.OPENAI_API_KEY }}}}
              prompt-file: .github/agent-review/review-prompt.md
              # Set this explicitly: with neither sandbox nor permission-profile, the action falls back to
              # workspace-write. (action.yml now labels sandbox "legacy" and prefers permission-profile: ":read-only".)
              sandbox: read-only
              output-schema-file: .github/agent-review/review.schema.json

      post-comments:
        needs: [claude-review, codex-review]
        if: ${{{{ always() && github.event.pull_request.draft == false }}}}
        runs-on: ubuntu-latest
        permissions:
          contents: read
          pull-requests: write
        steps:
          - uses: actions/checkout@v6
            with:
              ref: ${{{{ github.event.pull_request.base.sha }}}}
              persist-credentials: false
          - name: Post one deduplicated review
            env:
              GH_TOKEN: ${{{{ github.token }}}}
              REPO: ${{{{ github.repository }}}}
              PR_NUMBER: ${{{{ github.event.pull_request.number }}}}
              HEAD_SHA: ${{{{ github.event.pull_request.head.sha }}}}
              CLAUDE_FINDINGS: ${{{{ needs.claude-review.outputs.findings }}}}
              CODEX_FINDINGS: ${{{{ needs.codex-review.outputs.findings }}}}
              # success | failure | cancelled | skipped: anything but success makes this job fail (no silent pass)
              CLAUDE_RESULT: ${{{{ needs.claude-review.result }}}}
              CODEX_RESULT: ${{{{ needs.codex-review.result }}}}
            run: python .github/agent-review/post_review.py
    """)


write(".github/workflows/agent-review.yml", build_workflow(REVIEW_SCHEMA))
print((ROOT / ".github/workflows/agent-review.yml").read_text()[:900], "...")

In [ ]:
MODEL_SECRETS = ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "CLAUDE_CODE_OAUTH_TOKEN", "CODEX_API_KEY")


def audit_workflow(path: Path, schema_file: Path) -> list[tuple[str, bool, str]]:
    wf = yaml.safe_load(path.read_text())
    triggers = wf.get("on", wf.get(True))      # PyYAML 1.1 reads the bare key `on` as True
    rows = [("triggers on pull_request", isinstance(triggers, dict) and "pull_request" in triggers, ""),
            ("top-level permissions default to none", wf.get("permissions") == {}, str(wf.get("permissions")))]
    for name, job in wf["jobs"].items():
        text = json.dumps(job)
        perms = job.get("permissions", {})
        writes = sorted(k for k, v in perms.items() if v == "write" and k != "id-token")
        # id-token: write lets claude-code-action swap an OIDC token for a Claude GitHub App token, whose
        # permissions this job's block does not scope. Passing github_token keeps the job on GITHUB_TOKEN.
        if perms.get("id-token") == "write" and any(
                "claude-code-action" in str(s.get("uses", "")) and "github_token" not in s.get("with", {})
                for s in job.get("steps", [])):
            writes.append("Claude GitHub App token")
        keys = [s for s in MODEL_SECRETS if s in text]
        if keys:
            rows.append((f"{name}: holds {keys} and cannot write", not writes, f"writes {writes}"))
        if writes:
            rows.append((f"{name}: can write {writes} and holds no model key", not keys, f"keys {keys}"))
        for step in job.get("steps", []):
            if str(step.get("uses", "")).startswith("actions/checkout"):
                rows.append((f"{name}: checkout does not persist credentials",
                             step.get("with", {}).get("persist-credentials") is False, ""))
    file_schema = json.loads(schema_file.read_text())
    claude_args = next(s["with"]["claude_args"] for s in wf["jobs"]["claude-review"]["steps"] if "claude-code-action" in str(s.get("uses")))
    inline = json.loads(re.search(r"--json-schema '(.+)'", claude_args, re.S).group(1))
    rows.append(("inline --json-schema equals the schema file (no drift)", inline == file_schema, ""))
    codex_with = next(s["with"] for s in wf["jobs"]["codex-review"]["steps"] if "codex-action" in str(s.get("uses")))
    rows.append(("Codex reads the same schema file in a read-only sandbox",
                 codex_with.get("output-schema-file") == str(schema_file.relative_to(ROOT)) and codex_with.get("sandbox") == "read-only", ""))
    post = wf["jobs"]["post-comments"]["steps"][0]["with"].get("ref", "")
    rows.append(("poster runs code from the base commit, not the PR", "base.sha" in post, post))
    return rows


show_checks(audit_workflow(ROOT / ".github/workflows/agent-review.yml", ROOT / ".github/agent-review/review.schema.json"))

> **Observe:** `claude-review` passes `github_token: ${{ secrets.GITHUB_TOKEN }}` and has no `id-token: write`. Without the token input, the Claude action uses `id-token: write` to fetch a Claude GitHub App token instead, and a job's `permissions:` block scopes only `GITHUB_TOKEN`, not an App's token. The audit therefore counts "`id-token: write` and no `github_token`" as write access. Everything that can write lives in `post-comments`, which never sees a model key.

> **Try it:** add `pull-requests: write` to `codex-review` (or put `OPENAI_API_KEY` into `post-comments`) and rerun the audit. Then delete the `github_token:` line and add `id-token: write` to `claude-review`, the setup from the action's quick-start examples, and see which check fails. Finally, change one enum in `REVIEW_SCHEMA` only in the workflow text and watch the drift check fail.

In [ ]:
# Live: review the PR diff with both agents (off by default).
def run_live_reviews() -> list[dict]:
    git_init()
    (ROOT / "src/shop/api/routes.py").write_text(NEW_ROUTES)
    found = []
    try:
        if RUN_CLAUDE and CLAUDE_BIN:
            p = subprocess.run(claude_review_argv(REVIEW_PROMPT, REVIEW_SCHEMA), cwd=ROOT, input=PR_DIFF,
                               capture_output=True, text=True, timeout=900)
            found += collect("claude", findings_from_claude(p.stdout))
        if RUN_CODEX and CODEX_BIN:
            env = {**os.environ}
            if os.getenv("OPENAI_API_KEY"):
                env["CODEX_API_KEY"] = os.environ["OPENAI_API_KEY"]   # this process only
            p = subprocess.run(codex_review_argv(REVIEW_PROMPT + "\nThe diff:\n" + PR_DIFF), cwd=ROOT, env=env,
                               capture_output=True, text=True, timeout=900)
            found += collect("codex", findings_from_codex(p.stdout))
    finally:
        (ROOT / "src/shop/api/routes.py").write_text(OLD_ROUTES)
    return found


if RUN_CLAUDE or RUN_CODEX:
    LIVE_FINDINGS = run_live_reviews()
    print(json.dumps(LIVE_FINDINGS, indent=1)[:2000])
else:
    print("[offline] live reviews skipped; sections 4.8+ use the sample findings")

## 4.8 PR comments that stay useful

Three deterministic filters before anything is posted: inline comments only on lines that are part of the diff; drop findings whose `evidence` is not the text of the cited line; skip findings already posted on an earlier commit (keyed on file, category and normalized evidence, which survive line shifts). Findings from both agents are merged first. [Theory](README.md#48-pr-comments-that-stay-useful)

In [ ]:
HUNK_RE = re.compile(r"^@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")


def parse_unified_diff(diff: str) -> dict[str, dict[int, str]]:
    """{file: {new_line_number: text}} for every added or context line GitHub lets you comment on."""
    files: dict[str, dict[int, str]] = {}
    current, line_no = None, 0
    for raw in diff.splitlines():
        if raw.startswith("+++ "):
            current = raw[6:] if raw.startswith("+++ b/") else None
            files.setdefault(current, {}) if current else None
        elif raw.startswith("--- "):
            continue
        elif (m := HUNK_RE.match(raw)):
            line_no = int(m.group(1))
        elif current is not None and raw[:1] in {"+", " "}:
            files[current][line_no] = raw[1:]
            line_no += 1
    return files


def norm(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().lower()


def finding_key(f: dict) -> str:
    return hashlib.sha256(f"{f['file']}|{f['category']}|{norm(f['evidence'])}".encode()).hexdigest()[:12]


SEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}


def merge_findings(findings: list[dict]) -> list[dict]:
    """One entry per key; keep the most severe wording and remember which agents found it."""
    merged: dict[str, dict] = {}
    for f in sorted(findings, key=lambda f: SEVERITY_RANK[f["severity"]]):
        k = finding_key(f)
        if k in merged:
            merged[k]["sources"] = sorted(set(merged[k]["sources"]) | {f.get("source", "?")})
        else:
            merged[k] = {**f, "sources": [f.get("source", "?")], "key": k}
    return list(merged.values())


def route_findings(findings: list[dict], diff_lines: dict[str, dict[int, str]], file_text: dict[str, str],
                   already_posted: set[str]) -> dict[str, list[dict]]:
    routed = {"inline": [], "summary": [], "dropped_unsupported": [], "skipped_duplicate": []}
    for f in merge_findings(findings):
        actual = file_text.get(f["file"], "").splitlines()
        on_line = 0 < f["line"] <= len(actual) and norm(f["evidence"]) and norm(f["evidence"]) in norm(actual[f["line"] - 1])
        if not on_line:
            routed["dropped_unsupported"].append(f)
        elif f["key"] in already_posted:
            routed["skipped_duplicate"].append(f)
        elif f["line"] in diff_lines.get(f["file"], {}):
            routed["inline"].append(f)
        else:
            routed["summary"].append(f)
    return routed


def build_review(routed: dict[str, list[dict]], commit_sha: str) -> dict:
    """Body for POST /repos/{owner}/{repo}/pulls/{number}/reviews (dry run here)."""
    def body(f: dict) -> str:
        return (f"**[{f['severity']}] {f['category']}** ({' + '.join(f['sources'])}): {f['issue']}\n\n"
                f"Suggested fix: {f['suggested_fix']}\n<!-- agent-review:{f['key']} -->")
    summary = [f"- `{f['file']}:{f['line']}` [{f['severity']}] {f['issue']} <!-- agent-review:{f['key']} -->"
               for f in routed["summary"]]
    return {
        "commit_id": commit_sha,
        "event": "COMMENT",
        "body": "Agent review: {} inline, {} outside the diff, {} unsupported dropped, {} already reported.".format(
            len(routed["inline"]), len(routed["summary"]), len(routed["dropped_unsupported"]), len(routed["skipped_duplicate"]))
            + ("\n\nOutside this diff:\n" + "\n".join(summary) if summary else ""),
        "comments": [{"path": f["file"], "line": f["line"], "side": "RIGHT", "body": body(f)} for f in routed["inline"]],
    }


def gate(routed: dict[str, list[dict]], fail_on: frozenset = frozenset({"critical", "high"})) -> int:
    """CI exit code: 1 if any new or still-open inline finding is blocking, else 0."""
    blocking = [f for f in routed["inline"] + routed["skipped_duplicate"] if f["severity"] in fail_on]
    return 1 if blocking else 0


FILES_AFTER = {R: NEW_ROUTES, "src/shop/pricing.py": (ROOT / "src/shop/pricing.py").read_text()}
routed = route_findings(ALL_FINDINGS, parse_unified_diff(PR_DIFF), FILES_AFTER, already_posted=set())
for bucket, items in routed.items():
    print(f"{bucket:20}", [f"{f['file'].split('/')[-1]}:{f['line']} {f['category']} {f['sources']}" for f in items])
review = build_review(routed, commit_sha="abc1234")
print(json.dumps(review, indent=1)[:1200])
print("gate exit code:", gate(routed))

The author pushes a second commit that fixes the float math only. The re-review prompt includes the previous findings; the poster also remembers what it already posted (the hidden `agent-review:<key>` marker in each inline comment and in each summary line of the review body).

In [ ]:
FIXED_ROUTES = NEW_ROUTES.replace('item["price_cents"] * 1.0', 'item["price_cents"]')
PR_DIFF_2 = unified(OLD_ROUTES, FIXED_ROUTES, R)
posted_keys = {f["key"] for f in routed["inline"] + routed["summary"]}   # what the first review posted
rereview_prompt = REVIEW_PROMPT + "\nPrevious findings:\n" + json.dumps(
    [{k: f[k] for k in ("file", "line", "category", "issue")} for f in routed["inline"]], indent=1)
# SAMPLE re-review output: the KeyError issue is still there (line unchanged), the float issue is gone.
second = [{**F_RAISE, "source": "claude"}, {**F_NOTEST, "source": "codex", "line": line_of(FIXED_ROUTES, "def bulk_quote")}]
routed2 = route_findings(second, parse_unified_diff(PR_DIFF_2), {R: FIXED_ROUTES}, already_posted=posted_keys)
print("new inline comments:", len(routed2["inline"]), "| already reported, still open:",
      [f["issue"][:50] for f in routed2["skipped_duplicate"]])
print("gate exit code:", gate(routed2), "(still blocked by the open high-severity finding)")

The same functions become the repo's `post_review.py`. Writing them out with `inspect.getsource` keeps one copy of the logic; the `main()` below only adds the GitHub calls and runs as a dry run unless `GH_TOKEN` is set.

In [ ]:
POST_MAIN = '''

def main() -> int:
    """Merge both agents' findings, filter them against the PR diff, post one review."""
    import urllib.error
    import urllib.parse
    import urllib.request
    repo, number, sha = os.environ["REPO"], os.environ["PR_NUMBER"], os.environ["HEAD_SHA"]
    token = os.environ.get("GH_TOKEN")
    api = f"https://api.github.com/repos/{repo}/pulls/{number}"

    def call(url: str, accept: str = "application/vnd.github+json", data: bytes | None = None):
        req = urllib.request.Request(url, data=data, method="POST" if data else "GET", headers={
            "Accept": accept, "Authorization": f"Bearer {token}", "X-GitHub-Api-Version": "2022-11-28"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.read().decode()

    findings, incomplete = [], []
    for source in ("claude", "codex"):
        raw = os.environ.get(f"{source.upper()}_FINDINGS", "").strip()
        result = os.environ.get(f"{source.upper()}_RESULT")      # needs.<job>.result; unset only in local dry runs
        try:
            got = json.loads(raw)["findings"] if raw else None
        except (ValueError, KeyError, TypeError):
            got = None
        if result not in (None, "success") or not isinstance(got, list):
            # a failed, cancelled or skipped reviewer, or unusable output, is NOT "no findings"
            incomplete.append(f"{source} (job result: {result or 'unknown'}, output: {'valid' if isinstance(got, list) else 'missing or invalid'})")
            print(f"warning: no usable review from {source}")
        if isinstance(got, list):
            findings += [{**f, "source": source} for f in got]
    diff_lines = parse_unified_diff(call(api, accept="application/vnd.github.diff"))
    # Keys already posted: inline comments and earlier review bodies (the outside-the-diff summary lines).
    # First page only (100 of each). For PRs with longer review history, follow the Link header to paginate.
    earlier = json.loads(call(api + "/comments?per_page=100")) + json.loads(call(api + "/reviews?per_page=100"))
    posted = {k for c in earlier for k in re.findall(r"agent-review:([0-9a-f]{12})", c.get("body") or "")}

    def file_at_head(path: str) -> str:
        try:
            return call(f"https://api.github.com/repos/{repo}/contents/{urllib.parse.quote(path)}?ref={sha}",
                        accept="application/vnd.github.raw+json")
        except urllib.error.HTTPError:   # deleted or invented path: the evidence check drops the finding
            return ""

    # Every file a finding cites, not only the diff's files: out-of-diff findings need their text too.
    files = {path: file_at_head(path) for path in sorted({f["file"] for f in findings})}
    routed = route_findings(findings, diff_lines, files, posted)
    review = build_review(routed, sha)
    if incomplete:
        review["body"] += "\\n\\nIncomplete review, this check fails: " + "; ".join(incomplete)
    if os.environ.get("DRY_RUN") or not token:
        print(json.dumps(review, indent=2))
    elif review["comments"] or routed["summary"] or incomplete:
        call(api + "/reviews", data=json.dumps(review).encode())
    return 1 if incomplete else gate(routed)


if __name__ == "__main__":
    raise SystemExit(main())
'''

header = '"""Post one deduplicated review from both agents\' findings (generated by the capstone notebook)."""\nimport hashlib\nimport json\nimport os\nimport re\n\n'
try:
    parts = [textwrap.dedent(inspect.getsource(fn)) for fn in
             (parse_unified_diff, norm, finding_key, merge_findings, route_findings, build_review, gate)]
    consts = 'HUNK_RE = re.compile(r"^@@ -\\d+(?:,\\d+)? \\+(\\d+)(?:,\\d+)? @@")\nSEVERITY_RANK = {"critical": 0, "high": 1, "medium": 2, "low": 3}\n\n'
    write(".github/agent-review/post_review.py", header + consts + "\n\n".join(parts) + POST_MAIN)
    ast.parse((ROOT / ".github/agent-review/post_review.py").read_text())
    print("post_review.py written and parses:", (ROOT / ".github/agent-review/post_review.py").stat().st_size, "bytes")
except (OSError, TypeError) as e:   # inspect.getsource can fail in unusual kernels
    print("could not extract function sources:", e)

> **Observe:** the hallucinated `eval()` finding is dropped (its evidence is not on line 3), the `pricing.py` finding is real but outside the diff so it goes to the summary, and the float finding reported by **both** agents becomes **one** comment credited to both. The poster also **fails closed**: `post-comments` runs with `always()` so it can report, but if either reviewer job's `needs.<job>.result` is not `success` (failure, cancelled, skipped) or its output is missing or unparseable, `main()` says so in the review body and exits 1. A required check that passes when no review happened would be worse than no check.

> **Try it:** change `F_OUTSIDE["line"]` to the wrong line number and rerun the routing cell. Then think about what a large PR needs on top of this: per-file passes plus one cross-file integration pass.

## 4.9 Test generation without duplicates

Generated tests repeat existing scenarios unless the agent sees the existing suite. Build an inventory with `ast`, put it in the prompt, and still filter the output, because an instruction is not a guarantee. [Theory](README.md#49-test-generation-without-duplicates)

In [ ]:
def test_inventory(root: Path = ROOT) -> list[dict]:
    rows = []
    for path in sorted(p for p in root.rglob("test_*.py") if "__pycache__" not in p.parts):
        tree_ = ast.parse(path.read_text())
        for cls in (n for n in tree_.body if isinstance(n, ast.ClassDef)):
            for fn in (n for n in cls.body if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")):
                rows.append({"id": f"{path.relative_to(root)}::{cls.name}::{fn.name}",
                             "name": fn.name, "doc": ast.get_docstring(fn) or ""})
    return rows


INVENTORY = test_inventory()
for t in INVENTORY:
    print(f"{t['id']:58} {t['doc']}")

TESTGEN_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "required": ["tests_added", "skipped_as_duplicate", "verify_exit_code"],
    "properties": {
        "tests_added": {"type": "array", "items": {
            "type": "object", "additionalProperties": False, "required": ["file", "test_name", "scenario"],
            "properties": {"file": {"type": "string"}, "test_name": {"type": "string"}, "scenario": {"type": "string"}}}},
        "skipped_as_duplicate": {"type": "array", "items": {"type": "string"}},
        "verify_exit_code": {"type": "integer"},
    },
}
assert not strict_problems(TESTGEN_SCHEMA)
write(".github/agent-review/testgen.schema.json", json.dumps(TESTGEN_SCHEMA, indent=2) + "\n")

TESTGEN_PROMPT = ("Add unittest cases for src/shop/pricing.py following docs/agents/testing.md. "
                  "These tests already exist; do not repeat their scenarios:\n"
                  + "\n".join(f"- {t['id']}: {t['doc']}" for t in INVENTORY)
                  + "\nRun `python scripts/verify.py` at the end and report its exit code.")

claude_testgen_argv = ["claude", "-p", TESTGEN_PROMPT, "--output-format", "json",
                       "--json-schema", json.dumps(TESTGEN_SCHEMA, separators=(",", ":")),
                       "--permission-mode", "acceptEdits",
                       "--allowedTools", "Read,Grep,Glob,Edit,Write,Bash(python scripts/verify.py *)",
                       "--max-turns", "15", "--max-budget-usd", MAX_BUDGET_USD, "--model", CLAUDE_MODEL]
codex_testgen_argv = ["codex", "exec", "--json", "--sandbox", "workspace-write", "--model", OPENAI_MODEL,
                      "--output-schema", ".github/agent-review/testgen.schema.json", TESTGEN_PROMPT]
print("\n$", shlex.join(claude_testgen_argv)[:200], "...")
print("$", shlex.join(codex_testgen_argv)[:200], "...")

In [ ]:
def words(text: str) -> set[str]:
    stop = {"test", "a", "an", "the", "is", "with", "any", "off", "takes", "of", "to", "stays", "regardless"}
    return {w for w in re.findall(r"[a-z0-9]+", text.lower().replace("_", " ")) if w not in stop}


def is_duplicate(proposed: dict, inventory: list[dict], threshold: float = 0.6) -> str | None:
    """Existing test ID that already covers the proposed scenario, if any (name match or high word overlap)."""
    p = words(proposed["test_name"] + " " + proposed["scenario"])
    for t in inventory:
        if proposed["test_name"] == t["name"]:
            return t["id"]
        e = words(t["name"] + " " + t["doc"])
        if p and e and len(p & e) / len(p | e) >= threshold:
            return t["id"]
    return None


SAMPLE_PROPOSED = [   # what a generator might return without seeing the inventory
    {"file": "tests/test_pricing.py", "test_name": "test_save10_discount", "scenario": "SAVE10 takes 10% off"},
    {"file": "tests/test_pricing.py", "test_name": "test_lowercase_code", "scenario": "codes match regardless of case"},
    {"file": "tests/test_pricing.py", "test_name": "test_unknown_code_keeps_price", "scenario": "an unknown code leaves the price unchanged"},
    {"file": "tests/test_pricing.py", "test_name": "test_negative_price_rejected", "scenario": "a negative price raises ValueError"},
]
for p in SAMPLE_PROPOSED:
    dup = is_duplicate(p, INVENTORY)
    print(f"{'REJECT' if dup else 'keep  '} {p['test_name']:34} {('duplicates ' + dup) if dup else ''}")

> **Observe:** `test_lowercase_code` has a new name but repeats `test_code_is_case_insensitive`; the word-overlap check catches it. Word overlap is a cheap heuristic: in a real pipeline, show the rejected pairs to a reviewer rather than silently discarding them.

> **Try it:** raise `threshold` to 0.8 and watch the renamed duplicate (`test_lowercase_code`) slip through. A stricter threshold means fewer false rejections but more repeated scenarios; that is the trade-off you tune.

## 4.10 The verification contract and evidence

The contract has three layers: the human rules in `AGENTS.md`, a machine-readable JSON contract, and a validator that checks an agent's evidence bundle against real `verify.py` output. Here a scripted "agent" performs the fix test-first, so you can see a failed attempt reported honestly before the verified one. [Theory](README.md#410-the-verification-contract-and-evidence)

In [ ]:
CONTRACT = {
    "version": 1,
    "required_commands": ["python scripts/verify.py --evidence evidence/verify.json"],
    "pass_criteria": {"exit_code": 0, "tests_failed": 0, "tests_errors": 0,
                      "min_tests_run": "baseline", "max_new_skips": 0, "tests_removed": 0},
    "evidence_fields": ["task", "status", "files_changed", "commands", "tests", "baseline_tests_run",
                        "tests_added", "tests_removed", "risks", "attempts"],
    "on_failure": {"max_fix_attempts": 2, "action": "stop and report status 'failed' with failing test IDs",
                   "never": ["delete or skip a test", "weaken an assertion", "edit this contract"]},
}
write("docs/agents/verification-contract.json", json.dumps(CONTRACT, indent=2) + "\n")


def validate_evidence(ev: dict, contract: dict, baseline: dict, verify_report: dict) -> list[str]:
    """Contract violations in an agent's evidence bundle, cross-checked with the real verify report."""
    v = [f"missing field: {k}" for k in contract["evidence_fields"] if k not in ev]
    if v:
        return v
    crit = contract["pass_criteria"]
    ran = {c["cmd"]: c["exit_code"] for c in ev["commands"]}
    for cmd in contract["required_commands"]:
        if cmd not in ran:
            v.append(f"required command not run: {cmd}")
    removed = sorted(set(baseline["test_ids"]) - set(verify_report["test_ids"]))
    if ev["status"] == "verified":
        if any(code != crit["exit_code"] for code in ran.values()) or verify_report["exit_code"] != 0:
            v.append("status 'verified' but a command failed")
        if verify_report["tests"]["failed"] > crit["tests_failed"] or verify_report["tests"]["errors"] > crit["tests_errors"]:
            v.append("status 'verified' with failing tests")
        if verify_report["tests"]["run"] < baseline["tests"]["run"]:
            v.append(f"fewer tests than baseline ({verify_report['tests']['run']} < {baseline['tests']['run']})")
        if verify_report["tests"]["skipped"] - baseline["tests"]["skipped"] > crit["max_new_skips"]:
            v.append("new skipped tests")
    elif ev["status"] == "failed":
        if not verify_report["failing"]:
            v.append("status 'failed' must name the failing tests")
    else:
        v.append(f"unknown status {ev['status']!r}")
    if removed or ev["tests_removed"]:
        v.append(f"tests removed: {removed or ev['tests_removed']}")
    if ev["attempts"] > contract["on_failure"]["max_fix_attempts"]:
        v.append("more fix attempts than the contract allows")
    if ev["tests"] != verify_report["tests"]:
        v.append("reported test counts do not match verify.json")
    return v


print(json.dumps(CONTRACT["pass_criteria"]), "| baseline:", BASELINE["tests"])

In [ ]:
PRICING_BEFORE = (ROOT / "src/shop/pricing.py").read_text()
TESTS_BEFORE = (ROOT / "tests/test_pricing.py").read_text()
REGRESSION_TEST = '''
    def test_unknown_code_keeps_price(self):
        """An unknown discount code leaves the price unchanged instead of raising."""
        self.assertEqual(apply_discount(1000, "WELCOME"), 1000)
'''

# Attempt 1 (test first): add the regression test only. It must fail before the fix.
(ROOT / "tests/test_pricing.py").write_text(TESTS_BEFORE + REGRESSION_TEST)
attempt1 = run_verify()
print("attempt 1 failing:", attempt1["failing"])

# Attempt 2: fix the code.
(ROOT / "src/shop/pricing.py").write_text(PRICING_BEFORE.replace("DISCOUNT_PERCENT[code.upper()]", "DISCOUNT_PERCENT.get(code.upper(), 0)"))
attempt2 = run_verify()
print("attempt 2:", attempt2["tests"], "exit", attempt2["exit_code"])

In [ ]:
def files_changed(before: dict[str, str], root: Path = ROOT) -> list[str]:
    return sorted(p for p, text in before.items() if (root / p).read_text() != text)


EVIDENCE = {
    "task": "Unknown discount codes must not crash a quote",
    "status": "verified",
    "files_changed": files_changed({"src/shop/pricing.py": PRICING_BEFORE, "tests/test_pricing.py": TESTS_BEFORE}),
    "commands": [{"cmd": CONTRACT["required_commands"][0], "exit_code": attempt2["exit_code"]}],   # final attempt
    "tests": attempt2["tests"],
    "baseline_tests_run": BASELINE["tests"]["run"],
    "tests_added": sorted(set(attempt2["test_ids"]) - set(BASELINE["test_ids"])),
    "tests_removed": sorted(set(BASELINE["test_ids"]) - set(attempt2["test_ids"])),
    "risks": ["Unknown codes now leave the price unchanged without telling the caller; surfacing them is a follow-up."],
    "attempts": 2,
}
print(json.dumps(EVIDENCE, indent=1))
print("violations:", validate_evidence(EVIDENCE, CONTRACT, BASELINE, attempt2) or "none ✓")

In [ ]:
# Two dishonest bundles the validator must catch.
cheat_status = {**EVIDENCE, "tests": attempt1["tests"]}            # claims 'verified' on the failing run
print("verified on a failing run ->", validate_evidence(cheat_status, CONTRACT, BASELINE, attempt1))

# Simulate an agent that "fixed" verification by deleting a test it could not make pass.
(ROOT / "tests/test_pricing.py").write_text(TESTS_BEFORE.replace('    def test_zero_price', '    def _disabled_zero_price'))
shrunk = run_verify()
cheat_delete = {**EVIDENCE, "tests": shrunk["tests"]}
print("deleted a test ->", validate_evidence(cheat_delete, CONTRACT, BASELINE, shrunk))
(ROOT / "tests/test_pricing.py").write_text(TESTS_BEFORE + REGRESSION_TEST)   # restore the honest state
FINAL = run_verify()

In [ ]:
def pr_description(ev: dict, verify_report: dict) -> str:
    lines = [f"## {ev['task']}", "", f"**Status:** {ev['status']} · attempts: {ev['attempts']}", "",
             "### Files changed", *[f"- `{f}`" for f in ev["files_changed"]], "",
             "### Verification", *[f"- `{c['cmd']}` → exit {c['exit_code']}" for c in ev["commands"]],
             f"- Tests: {verify_report['tests']['run']} run (baseline {ev['baseline_tests_run']}), "
             f"{verify_report['tests']['failed']} failed, {verify_report['tests']['skipped']} skipped", "",
             "### Tests added", *([f"- `{t}`" for t in ev["tests_added"]] or ["- none"]), "",
             "### Risks and rollback", *[f"- {r}" for r in ev["risks"]], "- Rollback: revert this PR; no data migration involved."]
    return "\n".join(lines)


write("evidence/bundle.json", json.dumps(EVIDENCE, indent=2))
print(pr_description(EVIDENCE, FINAL))

> **Observe:** the evidence reports **2 attempts** and only the passing command, but the validator still cross-checks every number against the real `verify.json`. Claims are never trusted on their own.

**Live (optional):** let a real agent do the same task under the same contract. The prompt names the skill and the contract; the output schema makes the evidence machine-checkable.

In [ ]:
EVIDENCE_SCHEMA = {
    "type": "object", "additionalProperties": False, "required": CONTRACT["evidence_fields"],
    "properties": {
        "task": {"type": "string"}, "status": {"type": "string", "enum": ["verified", "failed"]},
        "files_changed": {"type": "array", "items": {"type": "string"}},
        "commands": {"type": "array", "items": {"type": "object", "additionalProperties": False,
                                                "required": ["cmd", "exit_code"],
                                                "properties": {"cmd": {"type": "string"}, "exit_code": {"type": "integer"}}}},
        "tests": {"type": "object", "additionalProperties": False, "required": ["run", "failed", "errors", "skipped"],
                  "properties": {k: {"type": "integer"} for k in ("run", "failed", "errors", "skipped")}},
        "baseline_tests_run": {"type": "integer"},
        "tests_added": {"type": "array", "items": {"type": "string"}},
        "tests_removed": {"type": "array", "items": {"type": "string"}},
        "risks": {"type": "array", "items": {"type": "string"}},
        "attempts": {"type": "integer"},
    },
}
assert not strict_problems(EVIDENCE_SCHEMA) and not validate(EVIDENCE, EVIDENCE_SCHEMA)
write(".github/agent-review/evidence.schema.json", json.dumps(EVIDENCE_SCHEMA, indent=2) + "\n")

FIX_PROMPT = ("Task: unknown discount codes must not crash apply_discount; they should leave the price unchanged. "
              "Work test-first. Follow the Definition of done in AGENTS.md and use the verify-change skill. "
              f"The baseline has {BASELINE['tests']['run']} tests. Return the evidence bundle.")
live_fix = {
    "claude": ["claude", "-p", FIX_PROMPT, "--output-format", "json", "--json-schema", json.dumps(EVIDENCE_SCHEMA, separators=(",", ":")),
               "--permission-mode", "acceptEdits", "--allowedTools", "Read,Grep,Glob,Edit,Write,Bash(python scripts/verify.py *)",
               "--max-turns", "20", "--max-budget-usd", MAX_BUDGET_USD, "--model", CLAUDE_MODEL],
    "codex": ["codex", "exec", "--json", "--sandbox", "workspace-write", "--model", OPENAI_MODEL,
              "--output-schema", ".github/agent-review/evidence.schema.json", FIX_PROMPT],
}
FIXED_STATE = {p: (ROOT / p).read_text() for p in ("src/shop/pricing.py", "tests/test_pricing.py")}
for tool, enabled, binary in (("claude", RUN_CLAUDE, CLAUDE_BIN), ("codex", RUN_CODEX, CODEX_BIN)):
    if not (enabled and binary):
        print(f"[offline] {tool}: $", shlex.join(live_fix[tool])[:160], "...")
        continue
    (ROOT / "src/shop/pricing.py").write_text(PRICING_BEFORE)            # reset to the buggy state
    (ROOT / "tests/test_pricing.py").write_text(TESTS_BEFORE)
    git_init()
    env = {**os.environ, **({"CODEX_API_KEY": os.environ["OPENAI_API_KEY"]} if tool == "codex" and os.getenv("OPENAI_API_KEY") else {})}
    p = subprocess.run(live_fix[tool], cwd=ROOT, env=env, capture_output=True, text=True, timeout=1200)
    ev = findings_from_claude(p.stdout) if tool == "claude" else findings_from_codex(p.stdout)
    report = run_verify()   # never trust the agent's numbers: rerun verification yourself
    print(tool, "violations:", validate_evidence(ev, CONTRACT, BASELINE, report) or "none ✓")
for p, text in FIXED_STATE.items():   # put the notebook's verified state back for the audit below
    (ROOT / p).write_text(text)

## Mini-project: `team_kit_audit()`

One command that runs every acceptance check from the README (R1–R10) against the repo and prints the acceptance table. In a real team this runs in CI on every change to agent configuration, so drift between the two agents' setups is caught by a failing build instead of by a confused developer.

In [ ]:
def check_plan_record() -> list[tuple[str, bool, str]]:
    modes = {t["id"]: choose_mode(t)[0] for t in TASKS}
    return [("plan-vs-direct decisions recorded for the three Exercise 2 tasks",
             modes == {"bugfix": "direct", "migration": "plan", "feature": "plan"}, str(modes))]


def check_comments() -> list[tuple[str, bool, str]]:
    r = route_findings(ALL_FINDINGS, parse_unified_diff(PR_DIFF), FILES_AFTER, set())
    r2 = route_findings(second, parse_unified_diff(PR_DIFF_2), {R: FIXED_ROUTES}, {f["key"] for f in r["inline"]})
    return [("unsupported findings dropped", [f["line"] for f in r["dropped_unsupported"]] == [F_GHOST["line"]], ""),
            ("out-of-diff findings go to the summary", [f["file"] for f in r["summary"]] == ["src/shop/pricing.py"], ""),
            ("cross-agent duplicates merged", any(len(f["sources"]) == 2 for f in r["inline"]), ""),
            ("re-review posts nothing already posted", not ({f["key"] for f in r2["inline"]} & {f["key"] for f in r["inline"]}), "")]


def check_testgen() -> list[tuple[str, bool, str]]:
    kept = [p["test_name"] for p in SAMPLE_PROPOSED if not is_duplicate(p, INVENTORY)]
    return [("existing tests are in the generation prompt", all(t["id"] in TESTGEN_PROMPT for t in INVENTORY), ""),
            ("duplicate proposals rejected", kept == ["test_unknown_code_keeps_price", "test_negative_price_rejected"], str(kept))]


def check_verification(root: Path = ROOT) -> list[tuple[str, bool, str]]:
    report = run_verify(root, quiet=True)
    ev = json.loads((root / "evidence/bundle.json").read_text())
    return [("verify.py passes and writes evidence", report["exit_code"] == 0 and (root / "evidence/verify.json").exists(), ""),
            ("evidence bundle satisfies the contract", not validate_evidence(ev, CONTRACT, BASELINE, report), ""),
            ("validator rejects a deleted test", bool(validate_evidence(cheat_delete, CONTRACT, BASELINE, shrunk)), "")]


AUDIT = {
    "R1 one source of truth": check_layering,
    "R2 path-scoped conventions": check_scoped_rules,
    "R3 skills for both agents": check_skills,
    "R4 MCP, no secrets": check_mcp,
    "R5 enforced guardrails": check_guardrails,
    "R6 plan vs direct": check_plan_record,
    "R7 CI review workflow": lambda: audit_workflow(ROOT / ".github/workflows/agent-review.yml", ROOT / ".github/agent-review/review.schema.json")
                                     + [("review schema in the strict subset", not strict_problems(REVIEW_SCHEMA), "")],
    "R8 useful PR comments": check_comments,
    "R9 no duplicate tests": check_testgen,
    "R10 verified with evidence": check_verification,
}


def team_kit_audit(verbose: bool = False) -> int:
    failed = 0
    print(f"{'requirement':30} {'checks':>7}  result")
    for req, fn in AUDIT.items():
        try:
            rows = fn()
        except Exception as e:  # noqa: BLE001 - a missing or broken file is a failed check, not a crash
            rows = [(f"check crashed: {type(e).__name__}: {e}", False, "")]
        bad = [(n, d) for n, ok, d in rows if not ok]
        failed += bool(bad)
        print(f"{req:30} {len(rows):>7}  {'PASS' if not bad else 'FAIL'}")
        for n, d in bad if (bad or verbose) else []:
            print(f"{'':40}✗ {n} {d}")
    print("\nACCEPTED ✅" if not failed else f"\n{failed} requirement(s) failing")
    return 1 if failed else 0


exit_code = team_kit_audit()

In [ ]:
# Break something on purpose: a teammate pastes the money rule into CLAUDE.md "to be safe".
original = (ROOT / "CLAUDE.md").read_text()
(ROOT / "CLAUDE.md").write_text(original + "- Money is always integer cents (`int`). Never use `float` for prices.\n")
team_kit_audit()
(ROOT / "CLAUDE.md").write_text(original)

> **Try it:** break one thing per requirement (remove `persist-credentials: false`, rename the shared skill folder, set `network_access = true`, delete a test) and confirm the audit names the right requirement each time. That is how you know the acceptance criteria are actually tested.

## Recap

- `AGENTS.md` is the single source of truth; `CLAUDE.md` imports it and adds only Claude-specific notes. `AGENTS.md` itself has no imports, because Codex reads them as text.
- File-type conventions live once in `docs/agents/`; Claude loads them through symlinked path-scoped rules, Codex through a routing table. Codex never reads instruction files below its working directory.
- Shared skills live in `.agents/skills/` with spec-only frontmatter and are symlinked into `.claude/skills/`; isolation is `context: fork` in Claude and a read-only custom agent in Codex.
- MCP: team servers in `.mcp.json` / trusted `.codex/config.toml` with credentials from the environment; personal servers in the home directory. Claude Code blanks its own credential variables toward remote servers.
- Enforcement beats instructions: `deny` → `ask` → `allow` in Claude, most-restrictive `.rules` in Codex, sandboxes in both, managed settings for "never".
- CI: one strict schema for `--json-schema` and `--output-schema`, key-holding jobs cannot write, the writing job runs trusted code, and every comment is filtered against the real diff, the cited line and earlier reviews.
- Done means verified: a contract, real `verify.py` evidence, and a validator that does not take the agent's word for it.

Next: [homework](02_homework.ipynb)